### Ячейка 1: Импорты

In [65]:
# Ячейка 1: Импорты
import os
import sys
import warnings
from pathlib import Path
import traceback
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
import torch
import torchaudio
import audiomentations
from typing import Tuple, Optional, Dict, List # <<< ДОБАВЛЕН Tuple
# Интерактивные виджеты
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual, Layout, VBox, HBox, Output, AppLayout, Accordion
from IPython.display import display, clear_output, Audio

# Отключить предупреждения для чистоты вывода
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

print("--- Версии библиотек ---")
print("Librosa Version:", librosa.__version__)
print("Soundfile Version:", sf.__version__)
print("Audiomentations Version:", audiomentations.__version__)
print("Torchaudio Version:", torchaudio.__version__)
print("ipywidgets Version:", widgets.__version__)
print("-" * 25)

# Стиль графиков
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 8) # Размер по умолчанию
plt.rcParams['figure.max_open_warning'] = 50 # Увеличить лимит открытых фигур

--- Версии библиотек ---
Librosa Version: 0.10.2.post1
Soundfile Version: 0.13.1
Audiomentations Version: 0.40.0
Torchaudio Version: 2.5.1+cu121
ipywidgets Version: 8.1.5
-------------------------


### Ячейка 2: Настройка Путей и Загрузка Метаданных

In [66]:
# Ячейка 2: Настройка Путей и Загрузка Метаданных


# --- Конфигурация Путей ---
DATA_DIR = Path("./")
AUDIO_FOLDER_NAME = "morse_dataset/morse_dataset"
TRAIN_CSV_PATH = DATA_DIR / "train.csv"
BACKGROUND_NOISE_PATH = Path("./_background_noise_") # !!! УКАЖИТЕ ПУТЬ к папке с шумами !!!

# --- Проверка и Загрузка ---
audio_folder_path = DATA_DIR / AUDIO_FOLDER_NAME
if not audio_folder_path.is_dir():
    print(f"❌ ОШИБКА: Папка с аудио не найдена: {audio_folder_path}")
    # Можно добавить распаковку архива здесь, если нужно
    # raise FileNotFoundError(f"Папка с аудио не найдена: {audio_folder_path}")
    valid_files = []
else:
    try:
        train_df = pd.read_csv(TRAIN_CSV_PATH)
        # Создаем полный путь и проверяем наличие файлов
        train_df['full_path'] = train_df['id'].apply(lambda x: audio_folder_path / f"{x}")
        train_df['exists'] = train_df['full_path'].apply(lambda p: p.exists())
        valid_files_df = train_df[train_df['exists']].copy()
        valid_files = valid_files_df['full_path'].tolist()
        print(f"✅ Найдено {len(valid_files)} существующих аудиофайлов из {len(train_df)} записей в CSV.")
        if not valid_files:
            print("⚠️ Не найдено ни одного валидного аудиофайла!")
    except FileNotFoundError:
        print(f"❌ ОШИБКА: Файл {TRAIN_CSV_PATH} не найден.")
        valid_files = []
        valid_files_df = pd.DataFrame(columns=['id', 'message', 'full_path']) # Пустой DataFrame
    except Exception as e:
        print(f"❌ ОШИБКА при загрузке CSV или проверке файлов: {e}")
        valid_files = []
        valid_files_df = pd.DataFrame(columns=['id', 'message', 'full_path'])

# Проверка папки с шумами
if not BACKGROUND_NOISE_PATH.is_dir():
    print(f"⚠️ Предупреждение: Папка с фоновыми шумами не найдена: {BACKGROUND_NOISE_PATH}")
    print("   Аугментация 'AddBackgroundNoise' будет недоступна.")
    noise_files_available = False
else:
    noise_files = list(BACKGROUND_NOISE_PATH.glob('*.*'))
    if not noise_files:
        print(f"⚠️ Предупреждение: Папка с фоновыми шумами '{BACKGROUND_NOISE_PATH}' пуста.")
        noise_files_available = False
    else:
        print(f"✅ Найдено {len(noise_files)} файлов фонового шума.")
        noise_files_available = True

# Глобальная переменная для хранения текущего аудио и SR
current_audio_data = {'y': None, 'sr': None, 'path': None, 'y_aug': None}

✅ Найдено 30000 существующих аудиофайлов из 30000 записей в CSV.
✅ Найдено 23 файлов фонового шума.


### Ячейка 3: Виджет Выбора Файла

In [ ]:
# Ячейка 3: Виджет Выбора Файла

style = {'description_width': 'initial'}
layout_long = Layout(width='95%')

if not valid_files:
    print("Нет доступных файлов для выбора.")
    file_selector = widgets.HTML("Нет доступных файлов.")
    selected_file_path = None
else:
    # Используем ID файла как ключ для отображения
    file_options = {row['id']: row['full_path'] for _, row in valid_files_df.iterrows()}
    # Сортируем по ID для предсказуемости
    sorted_ids = sorted(file_options.keys())
    file_selector = widgets.Dropdown(
        options=[(file_id, file_options[file_id]) for file_id in sorted_ids],
        description='Выберите аудиофайл:',
        style=style,
        layout=layout_long,
        value=file_options[sorted_ids[0]] # Выбираем первый файл по умолчанию
    )
    selected_file_path = file_selector.value

# Виджет для вывода информации о файле
file_info_output = widgets.Output()

def display_file_info(selected_path):
    """Отображает информацию о выбранном файле."""
    with file_info_output:
        clear_output(wait=True)
        if selected_path and selected_path.exists():
            try:
                file_id = selected_path.stem
                message = valid_files_df[valid_files_df['full_path'] == selected_path]['message'].iloc[0]
                print(f"Выбран файл: {selected_path.name}")
                print(f"ID: {file_id}")
                print(f"Сообщение (из CSV): {message}")
                # Загрузка и отображение аудио плеера
                y, sr = sf.read(selected_path, dtype='float32')
                current_audio_data['y'] = y
                current_audio_data['sr'] = sr
                current_audio_data['path'] = selected_path
                current_audio_data['y_aug'] = None # Сбрасываем аугментированное аудио
                display(Audio(data=y, rate=sr))
            except Exception as e:
                print(f"Ошибка при загрузке или отображении информации о файле {selected_path.name}: {e}")
                current_audio_data['y'] = None
                current_audio_data['sr'] = None
                current_audio_data['path'] = None
        else:
            print("Файл не выбран или не существует.")
            current_audio_data['y'] = None
            current_audio_data['sr'] = None
            current_audio_data['path'] = None

# Связываем функцию с виджетом выбора файла
widgets.interactive_output(display_file_info, {'selected_path': file_selector})

# Первичный вызов для отображения информации о файле по умолчанию
if selected_file_path:
    display_file_info(selected_file_path)

# Отображаем виджет выбора файла и область вывода информации
display(file_selector)
display(file_info_output)

Dropdown(description='Выберите аудиофайл:', layout=Layout(width='95%'), options=(('1.opus', WindowsPath('morse…

Output()

### Ячейка 4: Функции Обработки и Визуализации

In [68]:
# Ячейка 4: Функции Обработки и Визуализации (v9)

from scipy.signal import medfilt
import pandas as pd # Убедимся, что pandas импортирован
import math # Для проверки NaN/inf

# --- Вспомогательные Функции ---

# <--- ИСПРАВЛЕНА ОШИБКА TypeError и ПЕРЕИМЕНОВАНА в v9 --->
def calculate_adaptive_mask_mean_v9(rms_values, sr, hop_length,
                                      threshold_smoothing_duration_s: float,
                                      threshold_offset: float,
                                      apply_median_filter: bool = True,
                                      median_filter_ms: int = 30,
                                      min_absolute_rms_threshold: float = 0.005):
    """Рассчитывает адаптивный порог (pandas) с МИНИМАЛЬНЫМ ЗНАЧЕНИЕМ и бинарную маску."""
    if rms_values is None or rms_values.size == 0: return None, None
    moving_avg_len_frames = int(threshold_smoothing_duration_s * sr / hop_length)
    if moving_avg_len_frames % 2 == 0: moving_avg_len_frames += 1
    if moving_avg_len_frames <= 0: moving_avg_len_frames = 1
    try:
        rms_series = pd.Series(rms_values); local_mean_series = rms_series.rolling(window=moving_avg_len_frames, center=True, min_periods=1).mean()
        local_mean = local_mean_series.values
        if np.isnan(local_mean).any() or np.isinf(local_mean).any(): global_mean = np.nanmean(rms_values); local_mean = np.nan_to_num(local_mean, nan=global_mean if np.isfinite(global_mean) else 0.0)
    except Exception as e_pd: global_mean = np.mean(rms_values); local_mean = np.full_like(rms_values, global_mean if np.isfinite(global_mean) else 0.0)
    adaptive_threshold_base = local_mean + threshold_offset; adaptive_threshold = np.maximum(adaptive_threshold_base, min_absolute_rms_threshold)
    binary_mask_raw = (rms_values >= adaptive_threshold).astype(int)
    if apply_median_filter and median_filter_ms > 0:
        filter_length_frames = int(median_filter_ms * sr / (1000 * hop_length)); filter_length_frames += 1 if filter_length_frames % 2 == 0 else 0; filter_length_frames = max(1, filter_length_frames)
        if filter_length_frames >= len(binary_mask_raw): binary_mask_filtered = binary_mask_raw
        else:
            try: binary_mask_filtered = medfilt(binary_mask_raw, kernel_size=filter_length_frames)
            except Exception as e_filt: binary_mask_filtered = binary_mask_raw
    else: binary_mask_filtered = binary_mask_raw
    binary_mask = binary_mask_filtered.astype(int)
    if not np.all(np.isfinite(adaptive_threshold)) or not np.all(np.isfinite(binary_mask)):
        adaptive_threshold_base = local_mean + threshold_offset; adaptive_threshold = np.maximum(adaptive_threshold_base, min_absolute_rms_threshold)
        adaptive_threshold = np.nan_to_num(adaptive_threshold, nan=min_absolute_rms_threshold); binary_mask = (rms_values >= adaptive_threshold).astype(int); binary_mask = np.nan_to_num(binary_mask, nan=0).astype(int)
        if not np.all(np.isfinite(adaptive_threshold)): return None, None
    return adaptive_threshold, binary_mask

# Ячейка 4: Функции Обработки и Визуализации (Часть 2: calculate_features_and_metrics_v10)

# --- НОВАЯ ФУНКЦИЯ РАСЧЕТА ПРИЗНАКОВ И МЕТРИК (v10) ---
def calculate_features_and_metrics_v10(
    y: np.ndarray, sr: int,
    n_fft: int, hop_length: int, n_mels: int,
    threshold_smoothing_duration_s: float, threshold_offset: float,
    min_absolute_rms_threshold: float, apply_median_filter: bool, median_filter_ms: int,
    normalization_type: str, pcen_gain: float, pcen_bias: float, pcen_power: float,
    pcen_time_constant: float, peak_freq_neighbors: int = 2
) -> Dict[str, Optional[np.ndarray]]:
    """
    Рассчитывает основные признаки, маску, нормализованные спектрограммы,
    а также данные для визуализации пиковой частоты (v10).

    Args:
        y: Входной аудиосигнал (NumPy array).
        sr: Частота дискретизации.
        n_fft, hop_length, n_mels: Параметры STFT/Mel.
        threshold_*: Параметры адаптивной маски.
        apply_median_filter, median_filter_ms: Параметры постобработки маски.
        normalization_type: Тип нормализации ('None', 'Z-score', 'PCEN', 'Log').
        pcen_*: Параметры PCEN.
        peak_freq_neighbors: Количество соседей для мини-спектрограммы пика.

    Returns:
        Словарь с рассчитанными данными или None в случае ошибки.
        Ключи: 'rms', 'adaptive_threshold', 'binary_mask', 'linear_amp_spec',
               'linear_db_spec', 'zscore_linear_spec', 'pcen_linear_spec',
               'mel_energy_sum', 'peak_freq_indices', 'peak_freq_amps',
               'peak_mini_spec'
    """
    # Инициализируем словарь результатов со значениями None
    results = {key: None for key in [
        'rms', 'adaptive_threshold', 'binary_mask', 'linear_amp_spec',
        'linear_db_spec', 'zscore_linear_spec', 'pcen_linear_spec',
        'mel_energy_sum', 'peak_freq_indices', 'peak_freq_amps',
        'peak_mini_spec'
    ]}
    PCEN_DEFAULT_EPS = 1e-6 # Эпсилон для PCEN
    ZSCORE_EPS = 1e-8     # Эпсилон для Z-score
    LOG_EPSILON = 1e-8    # Эпсилон для log(0)

    # Проверка входных данных
    if y is None or y.size == 0:
        print("Warning (calc_v10): Входной сигнал y пуст.")
        return results # Возвращаем пустой словарь

    try:
        # 1. RMS и Адаптивная Маска
        # print("Debug calc_v10: Calculating RMS...") # Отладка
        rms_values = librosa.feature.rms(y=y, frame_length=n_fft, hop_length=hop_length)[0]
        if rms_values is None or rms_values.size == 0:
            print("Warning (calc_v10): RMS calculation failed or returned empty.") # Отладка
            raise ValueError("RMS calculation failed")
        results['rms'] = rms_values
        # print(f"Debug calc_v10: RMS shape: {rms_values.shape}") # Отладка

        # print("Debug calc_v10: Calculating Adaptive Mask...") # Отладка
        adaptive_threshold, binary_mask = calculate_adaptive_mask_mean_v9(
            rms_values, sr, hop_length, threshold_smoothing_duration_s, threshold_offset,
            apply_median_filter, median_filter_ms, min_absolute_rms_threshold
        )
        if adaptive_threshold is None or binary_mask is None:
            print("Warning (calc_v10): Mask calculation failed.") # Отладка
            raise ValueError("Mask calculation failed")
        results['adaptive_threshold'] = adaptive_threshold
        results['binary_mask'] = binary_mask
        # print(f"Debug calc_v10: Mask shape: {binary_mask.shape}") # Отладка


        # 2. STFT и Линейная Амплитудная Спектрограмма
        # print("Debug calc_v10: Calculating STFT...") # Отладка
        S_complex = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
        linear_amp_spec = np.abs(S_complex)
        # print(f"Debug calc_v10: Linear Amp Spec shape: {linear_amp_spec.shape}") # Отладка

        if not np.all(np.isfinite(linear_amp_spec)):
            print("Warning (calc_v10): NaN/inf found in linear_amp_spec.") # Отладка
            raise ValueError("NaN/inf in linear_amp_spec")
        results['linear_amp_spec'] = linear_amp_spec
        # print("Debug calc_v10: Calculating Linear dB Spec...") # Отладка
        results['linear_db_spec'] = librosa.amplitude_to_db(linear_amp_spec, ref=np.max)

        # 3. Мел-спектрограмма и Суммарная Энергия
        # print("Debug calc_v10: Calculating Mel Spec...") # Отладка
        # Используем linear_amp_spec**2 для энергии, затем power=1.0 в melspectrogram
        mel_spec = librosa.feature.melspectrogram(
            S=linear_amp_spec**2, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels, power=1.0
        )
        # print(f"Debug calc_v10: Mel Spec shape: {mel_spec.shape}") # Отладка
        results['mel_energy_sum'] = np.sum(mel_spec, axis=0)
        # print(f"Debug calc_v10: Mel Energy Sum shape: {results['mel_energy_sum'].shape}") # Отладка
        if not np.all(np.isfinite(results['mel_energy_sum'])):
            print("Warning (calc_v10): NaN/inf found in mel_energy_sum.") # Отладка


        # 4. Нормализация
        # print(f"Debug calc_v10: Applying Normalization: {normalization_type}") # Отладка
        if normalization_type == 'Z-score':
            mean_per_bin = np.mean(linear_amp_spec, axis=1, keepdims=True)
            std_per_bin = np.std(linear_amp_spec, axis=1, keepdims=True)
            # Добавляем эпсилон к std для избежания деления на ноль
            results['zscore_linear_spec'] = (linear_amp_spec - mean_per_bin) / (std_per_bin + ZSCORE_EPS)
            if not np.all(np.isfinite(results['zscore_linear_spec'])):
                print("Warning (calc_v10): NaN/inf found in zscore_linear_spec.") # Отладка
        elif normalization_type == 'PCEN':
            # PCEN ожидает амплитуду или энергию, librosa.pcen работает с амплитудой
            # Умножаем на 2**15 для масштабирования, как часто делают для аудио
            results['pcen_linear_spec'] = librosa.pcen(
                linear_amp_spec * (2**15), sr=sr, hop_length=hop_length, gain=pcen_gain,
                bias=pcen_bias, power=pcen_power, time_constant=pcen_time_constant, eps=PCEN_DEFAULT_EPS
            )
            if not np.all(np.isfinite(results['pcen_linear_spec'])):
                print("Warning (calc_v10): NaN/inf found in pcen_linear_spec.") # Отладка
        elif normalization_type == 'Log':
             # Применяем логарифм к амплитуде, добавляя эпсилон для избежания log(0)
             results['pcen_linear_spec'] = np.log(linear_amp_spec + LOG_EPSILON) # Сохраняем в pcen_linear_spec для переиспользования в отрисовке
             if not np.all(np.isfinite(results['pcen_linear_spec'])):
                print("Warning (calc_v10): NaN/inf found in log_linear_spec.") # Отладка
        elif normalization_type == 'None':
             # Если нормализация None, просто используем linear_amp_spec для отрисовки
             # Можно сохранить в pcen_linear_spec или zscore_linear_spec для унификации
             # Сохраним в pcen_linear_spec, чтобы отрисовка могла использовать один ключ
             results['pcen_linear_spec'] = linear_amp_spec # Используем амплитуду напрямую
        else:
            print(f"Warning (calc_v10): Unknown normalization type: {normalization_type}") # Отладка
            # Если тип неизвестен, не применяем нормализацию, но и не возвращаем ошибку
            # results['pcen_linear_spec'] = linear_amp_spec # Fallback

        # 5. Расчет Данных для Пиковой Частоты
        # print("Debug calc_v10: Calculating Peak Frequency data...") # Отладка
        n_freq_bins, n_frames = linear_amp_spec.shape

        # Находим индекс максимальной частоты для каждого кадра
        # argmax возвращает индекс первого максимального значения, если их несколько
        peak_indices = np.argmax(linear_amp_spec, axis=0)
        results['peak_freq_indices'] = peak_indices
        # print(f"Debug calc_v10: Peak indices shape: {peak_indices.shape}") # Отладка

        # Извлекаем амплитуду на пиковой частоте для каждого кадра
        # Используем расширенное индексирование NumPy
        peak_amps = linear_amp_spec[peak_indices, np.arange(n_frames)]
        results['peak_freq_amps'] = peak_amps
        # print(f"Debug calc_v10: Peak amps shape: {peak_amps.shape}") # Отладка
        if not np.all(np.isfinite(peak_amps)):
             print("Warning (calc_v10): NaN/inf found in peak_freq_amps.") # Отладка


        # Создаем мини-спектрограмму вокруг пика
        num_neighbors = peak_freq_neighbors
        mini_spec_shape = (2 * num_neighbors + 1, n_frames)
        mini_spec = np.full(mini_spec_shape, np.nan, dtype=linear_amp_spec.dtype) # Заполняем NaN

        for t in range(n_frames):
            peak_k = peak_indices[t]
            for offset_idx, offset in enumerate(range(-num_neighbors, num_neighbors + 1)):
                current_k = peak_k + offset
                # Проверяем границы частотных бинов
                if 0 <= current_k < n_freq_bins:
                    mini_spec[offset_idx, t] = linear_amp_spec[current_k, t]
                # else: оставляем NaN

        # Заменяем NaN (на краях, где соседей нет) на 0 для визуализации
        mini_spec = np.nan_to_num(mini_spec, nan=0.0)
        results['peak_mini_spec'] = mini_spec
        # print(f"Debug calc_v10: Peak mini spec shape: {mini_spec.shape}") # Отладка
        if not np.all(np.isfinite(mini_spec)):
             print("Warning (calc_v10): NaN/inf found in peak_mini_spec after nan_to_num.") # Отладка

        # 6. Подгонка размеров маски (на всякий случай, если длины не совпали)
        # Длина маски должна совпадать с количеством кадров в спектрограмме
        n_frames_spec = linear_amp_spec.shape[1]
        if results['binary_mask'] is not None and len(results['binary_mask']) != n_frames_spec:
            # print(f"Debug calc_v10: Mask length ({len(results['binary_mask'])}) mismatch with spec frames ({n_frames_spec}). Resizing mask.") # Отладка
            mask = results['binary_mask']
            if len(mask) > n_frames_spec:
                results['binary_mask'] = mask[:n_frames_spec]
            elif len(mask) < n_frames_spec:
                # Паддинг нулями, если маска короче
                if mask.size > 0:
                    results['binary_mask'] = np.pad(mask, (0, n_frames_spec - len(mask)), mode='edge') # Можно 'constant' с 0
                else: # Если маска была пустой
                    results['binary_mask'] = np.zeros(n_frames_spec, dtype=int)
            # Также подгоняем adaptive_threshold, если он был рассчитан
            if results['adaptive_threshold'] is not None and len(results['adaptive_threshold']) != n_frames_spec:
                 thresh = results['adaptive_threshold']
                 if len(thresh) > n_frames_spec:
                     results['adaptive_threshold'] = thresh[:n_frames_spec]
                 elif len(thresh) < n_frames_spec:
                      if thresh.size > 0:
                          results['adaptive_threshold'] = np.pad(thresh, (0, n_frames_spec - len(thresh)), mode='edge') # Можно 'constant' с 0
                      else:
                          results['adaptive_threshold'] = np.zeros(n_frames_spec, dtype=float)


        # print("Debug calc_v10: Calculation finished.") # Отладка
        return results

    except Exception as e:
        print(f"❌ Ошибка в calculate_features_and_metrics_v10: {e}")
        # traceback.print_exc(limit=2) # Раскомментировать для детальной отладки
        # В случае любой ошибки, возвращаем словарь с None значениями
        return {key: None for key in results}

print("Функция calculate_features_and_metrics_v10 определена.")

# --- Основная Функция Визуализации (v9 - Добавлен график Mel Energy Sum) ---
# --- Обновленная Функция Визуализации (v10.1 - Компактная) ---



# Ячейка 4: Функции Обработки и Визуализации (Часть 3: Основная функ# Ячейка 4: Функции Обработки и Визуализации (Часть 3: plot_combined_representations_v10.3)



# --- Обновленная Функция Визуализации (v10.3 - Фикс aspect, Компактность, Отладка) ---
def plot_combined_representations_v10(
    results: Dict[str, Optional[np.ndarray]],
    y: np.ndarray, sr: int, y_aug: Optional[np.ndarray] = None,
    n_fft: int = 512, hop_length: int = 128, n_mels: int = 80,
    offset_for_title: Optional[float] = None,
    duration_for_title: Optional[float] = None,
    spec_before_aug: Optional[np.ndarray] = None, # Для SpecAugment
    spec_after_aug: Optional[np.ndarray] = None,  # Для SpecAugment
    spec_aug_title: str = "",
    vis_fmax: Optional[int] = None, title_suffix: str = "",
    peak_freq_neighbors: int = 2 # Количество соседей для мини-спектрограммы пика
):
    """
    Строит все графики вместе, используя данные из словаря results (v10.3).
    Включает графики пиковой частоты. Компактный вид.
    """
    # --- Извлекаем данные из словаря results ---
    linear_db_spec = results.get('linear_db_spec')
    zscore_linear_spec = results.get('zscore_linear_spec')
    pcen_linear_spec = results.get('pcen_linear_spec') # Используется также для Log нормализации
    mel_energy_sum = results.get('mel_energy_sum')
    rms_values = results.get('rms')
    adaptive_threshold = results.get('adaptive_threshold')
    binary_mask = results.get('binary_mask')
    peak_freq_indices = results.get('peak_freq_indices') # Индексы пиков
    peak_freq_amps = results.get('peak_freq_amps')       # Амплитуды пиков
    peak_mini_spec = results.get('peak_mini_spec')       # Мини-спектрограмма

    # --- Отладка v10.3: Проверка полученных данных ---
    # print(f"\n--- Debug plot_combined_representations_v10.3 ---")
    # print(f"  Получен словарь results с ключами: {list(results.keys())}")
    # for key in results:
    #     data = results[key]
    #     if data is not None:
    #         print(f"  '{key}': Shape={data.shape}, Dtype={data.dtype}, IsFinite={np.all(np.isfinite(data))}")
    #         if np.all(np.isfinite(data)) and data.size > 0:
    #              print(f"    Min={np.min(data):.4f}, Max={np.max(data):.4f}, Mean={np.mean(data):.4f}")
    #     else:
    #         print(f"  '{key}': is None")
    # print("--------------------------------------------------\n")
    # --- Конец Отладки ---


    # --- Определяем, какие графики рисовать и сколько их ---
    plot_indices = {}; current_plot_idx = 1; n_frames = 0

    # Графики, которые всегда должны быть, если данные доступны
    if y is not None and y.size > 0: plot_indices['waveform'] = current_plot_idx; current_plot_idx += 1
    if y_aug is not None and y_aug.size > 0: plot_indices['waveform_aug'] = current_plot_idx; current_plot_idx += 1

    # Спектрограммы (берем n_frames из первой доступной)
    if linear_db_spec is not None and linear_db_spec.size > 0: plot_indices['linear_db_spec'] = current_plot_idx; current_plot_idx += 1; n_frames = linear_db_spec.shape[1]
    if zscore_linear_spec is not None and zscore_linear_spec.size > 0: plot_indices['zscore_linear_spec'] = current_plot_idx; current_plot_idx += 1; n_frames = n_frames or zscore_linear_spec.shape[1]
    if pcen_linear_spec is not None and pcen_linear_spec.size > 0: plot_indices['pcen_linear_norm_spec'] = current_plot_idx; current_plot_idx += 1; n_frames = n_frames or pcen_linear_spec.shape[1] # Используется также для Log/None

    # Другие графики, зависящие от наличия данных
    if mel_energy_sum is not None and mel_energy_sum.size > 0: plot_indices['mel_energy_sum'] = current_plot_idx; current_plot_idx += 1; n_frames = n_frames or len(mel_energy_sum)
    if peak_freq_amps is not None and peak_freq_amps.size > 0: plot_indices['peak_freq_amps'] = current_plot_idx; current_plot_idx += 1; n_frames = n_frames or len(peak_freq_amps)
    if peak_mini_spec is not None and peak_mini_spec.size > 0: plot_indices['peak_mini_spec'] = current_plot_idx; current_plot_idx += 1; n_frames = n_frames or peak_mini_spec.shape[1]
    if rms_values is not None and rms_values.size > 0: plot_indices['rms_plot'] = current_plot_idx; current_plot_idx += 1; n_frames = n_frames or len(rms_values)
    if binary_mask is not None and binary_mask.size > 0: plot_indices['mask_plot'] = current_plot_idx; current_plot_idx += 1; n_frames = n_frames or len(binary_mask)

    # SpecAugment графики (зависят от наличия данных SpecAugment)
    if spec_before_aug is not None and spec_before_aug.size > 0: plot_indices['spec_before_aug'] = current_plot_idx; current_plot_idx += 1; n_frames = n_frames or spec_before_aug.shape[1]
    if spec_after_aug is not None and spec_after_aug.size > 0: plot_indices['spec_after_aug'] = current_plot_idx; current_plot_idx += 1; n_frames = n_frames or spec_after_aug.shape[1]


    num_total_plots = current_plot_idx - 1

    if num_total_plots == 0:
        print("Warning (plot_v10.3): Нет данных для отрисовки графиков.")
        return

    if n_frames == 0:
        print("Warning (plot_v10.3): Не удалось определить количество временных кадров.")
        # Попробуем оценить по длине аудио, если n_frames все еще 0
        if y is not None and y.size > 0 and hop_length > 0:
             n_frames = int(np.ceil(len(y) / hop_length))
             print(f"Warning (plot_v10.3): Оценка n_frames по длине аудио: {n_frames}")
        if n_frames == 0:
             print("Warning (plot_v10.3): Невозможно рассчитать временную ось. Пропуск отрисовки.")
             return


    # Расчет временной оси
    times = librosa.times_like(np.arange(n_frames), sr=sr, hop_length=hop_length, n_fft=n_fft)
    # Убедимся, что длина times совпадает с n_frames
    if len(times) != n_frames:
        # print(f"Debug plot_v10.3: Длина times ({len(times)}) не совпадает с n_frames ({n_frames}). Корректировка.") # Отладка
        if len(times) > n_frames: times = times[:n_frames]
        else: # Если times короче, что маловероятно при правильном расчете librosa
             warnings.warn(f"Длина times ({len(times)}) меньше n_frames ({n_frames}). Возможная ошибка в расчете временной оси.")
             # Попытка создать временную ось вручную
             times = np.arange(n_frames) * hop_length / sr


    # Настройка subplot'ов
    ncols = 1; nrows = num_total_plots
    fig_height = 2.5 * nrows # Сделаем графики еще более компактными
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(12, fig_height), sharex=False, squeeze=False)
    axes = axes.flatten() # Делаем массив осей плоским
    fig.suptitle(f"Объединенный Анализ Аудио (v10.3) {title_suffix}", fontsize=16, y=1.01)
    DB_CMAP = 'magma'; VAL_CMAP = 'viridis'; MASK_CMAP = plt.cm.get_cmap('binary', 2)

    # Вспомогательная функция vmin/vmax (без изменений)
    def get_percentile_vmin_vmax(data, p_low=1, p_high=99, skip_frames=0):
         if data is None or data.size == 0: return None, None
         if data.ndim == 2 and data.shape[1] > skip_frames: valid_data = data[:, skip_frames:]
         elif data.ndim == 1 and len(data) > skip_frames: valid_data = data[skip_frames:]
         else: valid_data = data
         if valid_data.size == 0: return None, None
         vmin = np.percentile(valid_data, p_low); vmax = np.percentile(valid_data, p_high)
         if vmin is None or vmax is None or np.isclose(vmin, vmax): vmin=0.0; vmax=1.0
         if not np.isfinite(vmin): vmin = np.nanmin(valid_data) if np.any(np.isfinite(valid_data)) else 0.0
         if not np.isfinite(vmax): vmax = np.nanmax(valid_data) if np.any(np.isfinite(valid_data)) else 1.0
         if np.isclose(vmin, vmax): vmin -= 1e-6; vmax += 1e-6
         return vmin, vmax

    current_ax_idx = 0 # Индекс для обращения к axes

    # --- Отрисовка ---

    # 1. Waveform Original
    if 'waveform' in plot_indices:
        ax = axes[current_ax_idx]; current_ax_idx += 1; is_last_plot = (current_ax_idx == num_total_plots)
        librosa.display.waveshow(y, sr=sr, ax=ax, color='blue', alpha=0.7)
        ax.set_title(f"Waveform (Original) | SR={sr}Hz | Len={len(y)/sr:.2f}s")
        ax.set_xlabel("Время (с)" if is_last_plot else None); ax.set_ylabel("Амплитуда"); ax.grid(True, linestyle='--', alpha=0.6)
        if not is_last_plot: ax.tick_params(labelbottom=False)

    # 2. Waveform Augmented
    if 'waveform_aug' in plot_indices and y_aug is not None:
        ax = axes[current_ax_idx]; current_ax_idx += 1; is_last_plot = (current_ax_idx == num_total_plots)
        librosa.display.waveshow(y_aug, sr=sr, ax=ax, color='red', alpha=0.7)
        ax.set_title(f"Waveform (Augmented)")
        ax.set_xlabel("Время (с)" if is_last_plot else None); ax.set_ylabel("Амплитуда"); ax.grid(True, linestyle='--', alpha=0.6)
        if not is_last_plot: ax.tick_params(labelbottom=False)

    # 3. Linear dB Spectrogram
    if 'linear_db_spec' in plot_indices and linear_db_spec is not None:
        ax = axes[current_ax_idx]; current_ax_idx += 1; is_last_plot = (current_ax_idx == num_total_plots)
        img = librosa.display.specshow(linear_db_spec, sr=sr, hop_length=hop_length, x_axis='time', y_axis='linear', ax=ax, fmax=vis_fmax, cmap=DB_CMAP, n_fft=n_fft)
        ax.set_title("Linear Spectrogram (dB)")
        ax.set_xlabel("Время (с)" if is_last_plot else None); ax.set_ylabel("Частота (Hz)"); fig.colorbar(img, ax=ax, format='%+2.0f dB')
        if not is_last_plot: ax.tick_params(labelbottom=False)

    # 4. Z-score Linear Spectrogram
    if 'zscore_linear_spec' in plot_indices and zscore_linear_spec is not None:
        ax = axes[current_ax_idx]; current_ax_idx += 1; is_last_plot = (current_ax_idx == num_total_plots)
        vmin, vmax = get_percentile_vmin_vmax(zscore_linear_spec, p_low=1, p_high=99)
        img = librosa.display.specshow(zscore_linear_spec, sr=sr, hop_length=hop_length, x_axis='time', y_axis='linear', ax=ax, fmax=vis_fmax, cmap=VAL_CMAP, vmin=vmin, vmax=vmax, n_fft=n_fft)
        ax.set_title("Z-score Normalized Linear Spectrogram")
        ax.set_xlabel("Время (с)" if is_last_plot else None); ax.set_ylabel("Частота (Hz)"); fig.colorbar(img, ax=ax, format='%.2f')
        if not is_last_plot: ax.tick_params(labelbottom=False)

    # 5. PCEN / Log / None Normalized Linear Spectrogram
    # Используем pcen_linear_spec для отрисовки, т.к. в него сохраняются результаты Log и None
    if 'pcen_linear_norm_spec' in plot_indices and pcen_linear_spec is not None:
        ax = axes[current_ax_idx]; current_ax_idx += 1; is_last_plot = (current_ax_idx == num_total_plots)
        # Определяем тип нормализации для заголовка и цветовой шкалы
        norm_title = "PCEN Normalized Linear Spectrogram"
        cmap_to_use = VAL_CMAP
        cbar_format = '%.3f'
        # Проверяем, какой тип нормализации был применен (нужно получить из виджетов или передать)
        # В текущей структуре виджеты доступны только в process_and_plot_combined_v10
        # Передадим тип нормализации как аргумент, или попробуем получить из results (если сохраним)
        # Пока используем PCEN как основной, если данные есть в pcen_linear_spec
        # Лучше передать тип нормализации явно из process_and_plot_combined_v10
        # Добавим аргумент normalization_type_applied в plot_combined_representations_v10
        # ... (пока пропустим, чтобы не усложнять)

        # Для PCEN и Z-score используем перцентили, для Linear/Log - min/max или другие
        # Простая логика vmin/vmax: если есть отрицательные значения (Z-score, Log), используем симметричную шкалу
        # Если только положительные (PCEN, Linear), используем 0 до max
        if np.min(pcen_linear_spec) < 0:
             abs_max = np.max(np.abs(pcen_linear_spec))
             vmin, vmax = -abs_max, abs_max
             cbar_format = '%.2f' # Z-score/Log обычно проще
             norm_title = "Z-score Normalized Linear Spectrogram" if 'zscore_linear_spec' in results and results['zscore_linear_spec'] is not None else "Log Normalized Linear Spectrogram"
        else:
             vmin, vmax = get_percentile_vmin_vmax(pcen_linear_spec, p_low=0.1, p_high=99.9, skip_frames=5)
             norm_title = "PCEN Normalized Linear Spectrogram" if 'pcen_linear_spec' in results and results['pcen_linear_spec'] is not None else "Linear Amplitude Spectrogram"
             cbar_format = '%.3f' if norm_title == "PCEN Normalized Linear Spectrogram" else '%.4f'


        img = librosa.display.specshow(pcen_linear_spec, sr=sr, hop_length=hop_length, x_axis='time', y_axis='linear', ax=ax, fmax=vis_fmax, cmap=cmap_to_use, vmin=vmin, vmax=vmax, n_fft=n_fft)
        ax.set_title(norm_title) # Используем определенный заголовок
        ax.set_xlabel("Время (с)" if is_last_plot else None); ax.set_ylabel("Частота (Hz)"); fig.colorbar(img, ax=ax, format=cbar_format)
        if not is_last_plot: ax.tick_params(labelbottom=False)

    # 6. Суммарная Мел-Энергия
    if 'mel_energy_sum' in plot_indices and mel_energy_sum is not None:
        ax = axes[current_ax_idx]; current_ax_idx += 1; is_last_plot = (current_ax_idx == num_total_plots)
        if len(mel_energy_sum) > n_frames: mel_energy_sum_plot = mel_energy_sum[:n_frames]
        elif len(mel_energy_sum) < n_frames: mel_energy_sum_plot = np.pad(mel_energy_sum, (0, n_frames - len(mel_energy_sum)), mode='edge')
        else: mel_energy_sum_plot = mel_energy_sum
        ax.plot(times, mel_energy_sum_plot, label='Сумма Мел-Энергии', color='green', alpha=0.8, linewidth=1.5)
        ax.set_title(f"Суммарная Энергия Мел-Спектрограммы (n_mels={n_mels})")
        ax.set_xlabel("Время (с)" if is_last_plot else None); ax.set_ylabel("Сумм. Энергия"); ax.legend(fontsize='small', loc='upper right'); ax.grid(True, linestyle='--', alpha=0.6)
        ax.set_xlim(times[0], times[-1])
        if not is_last_plot: ax.tick_params(labelbottom=False)

    # 7. Амплитуда Пиковой Частоты (Волновой график)
    if 'peak_freq_amps' in plot_indices and peak_freq_amps is not None:
        ax = axes[current_ax_idx]; current_ax_idx += 1; is_last_plot = (current_ax_idx == num_total_plots)
        if len(peak_freq_amps) > n_frames: peak_freq_amps_plot = peak_freq_amps[:n_frames]
        elif len(peak_freq_amps) < n_frames: peak_freq_amps_plot = np.pad(peak_freq_amps, (0, n_frames - len(peak_freq_amps)), mode='edge')
        else: peak_freq_amps_plot = peak_freq_amps
        ax.plot(times, peak_freq_amps_plot, label='Амплитуда пиковой частоты', color='darkorange', alpha=0.9, linewidth=1.5)
        ax.set_title("Амплитуда Самой Сильной Частоты в Кадре")
        ax.set_xlabel("Время (с)" if is_last_plot else None); ax.set_ylabel("Амплитуда"); ax.legend(fontsize='small', loc='upper right'); ax.grid(True, linestyle='--', alpha=0.6)
        ax.set_xlim(times[0], times[-1])
        if not is_last_plot: ax.tick_params(labelbottom=False)

            # 8. Мини-Спектрограмма Вокруг Пика (Цветной график) - ОТЛАДКА v10.5
    print(f"\n--- Debug plot v10.5: Проверка перед блоком 8 (Мини-спектрограмма) ---")
    # ... (предыдущие debug принты можно оставить или убрать) ...
    print(f"  peak_mini_spec не None? {'Да' if peak_mini_spec is not None else 'Нет'}")
    if peak_mini_spec is not None:
        print(f"  peak_mini_spec: Shape={peak_mini_spec.shape}, Dtype={peak_mini_spec.dtype}, IsFinite={np.all(np.isfinite(peak_mini_spec))}")
        if np.all(np.isfinite(peak_mini_spec)) and peak_mini_spec.size > 0:
             print(f"    Min={np.min(peak_mini_spec):.4f}, Max={np.max(peak_mini_spec):.4f}, Mean={np.mean(peak_mini_spec):.4f}")
    print("--------------------------------------------------------------------")


    if 'peak_mini_spec' in plot_indices and peak_mini_spec is not None and peak_mini_spec.size > 0 and np.all(np.isfinite(peak_mini_spec)):
        try:
            print(f"  >>> Вход в блок отрисовки мини-спектрограммы v10.5 (current_ax_idx={current_ax_idx})")
            ax = axes[current_ax_idx]; current_ax_idx += 1; is_last_plot = (current_ax_idx == num_total_plots)
            print(f"      Получены оси: {ax}")
            print(f"      Данные для отрисовки: peak_mini_spec.shape = {peak_mini_spec.shape}")

            # --- Рассчитываем vmin/vmax для адекватной шкалы ---
            vmin_mini, vmax_mini = get_percentile_vmin_vmax(peak_mini_spec, p_low=1, p_high=99)
            if vmin_mini is None: vmin_mini = 0.0 # Fallback
            if vmax_mini is None: vmax_mini = np.max(peak_mini_spec) if peak_mini_spec.size > 0 else 1.0 # Fallback
            print(f"      Рассчитаны vmin={vmin_mini:.4f}, vmax={vmax_mini:.4f}")

            # --- ИЗМЕНЕНО: Используем y_axis='linear' и передаем vmin/vmax ---
            print("      Попытка отрисовки через librosa.display.specshow с y_axis='linear'...")
            img = librosa.display.specshow(
                peak_mini_spec,
                x_axis='time',
                y_axis='linear', # <--- ИЗМЕНЕНО
                ax=ax,
                cmap='viridis',
                hop_length=hop_length,
                sr=sr,
                vmin=vmin_mini, # <--- ДОБАВЛЕНО
                vmax=vmax_mini  # <--- ДОБАВЛЕНО
            )
            print("      librosa.display.specshow выполнен.")
            fig.colorbar(img, ax=ax, format='%.4f')
            print("      colorbar добавлен.")
            # --- КОНЕЦ ИЗМЕНЕНИЙ ---

            # Добавляем линию на уровне пика (индекс peak_freq_neighbors соответствует смещению 0)
            # Поскольку y_axis='linear', нам нужно отобразить y-координату этого бина
            # (хотя для imshow было проще просто указать индекс)
            # Вместо этого оставим настройку yticks, она переопределит линейную шкалу
            # ax.axhline(y=peak_freq_neighbors, color='white', linestyle=':', linewidth=1.0, alpha=0.6) # Пока закомментируем, т.к. ось линейная

            ax.set_title(f"Мини-Спектрограмма Вокруг Пика (±{peak_freq_neighbors} бинов)")
            ax.set_xlabel("Время (с)" if is_last_plot else None); ax.set_ylabel("Смещение от Пика")

            # --- ОСТАВЛЯЕМ: Ручная настройка меток оси Y ---
            # Это должно переопределить линейную шкалу, которую нарисовал specshow
            num_rows = peak_mini_spec.shape[0]
            y_ticks = np.linspace(0, ax.get_ylim()[1], num_rows, endpoint=False) + (ax.get_ylim()[1] / num_rows / 2) # Центрируем метки в бинах
            # Убедимся, что количество тиков совпадает с количеством меток
            if len(y_ticks) != (2 * peak_freq_neighbors + 1):
                y_ticks = np.arange(2 * peak_freq_neighbors + 1) # Fallback на простые индексы

            y_tick_labels = [str(i - peak_freq_neighbors) for i in range(2 * peak_freq_neighbors + 1)]
            ax.set_yticks(y_ticks)
            ax.set_yticklabels(y_tick_labels)
            print("      Ручные метки оси Y установлены.")
            # --- КОНЕЦ НАСТРОЙКИ МЕТОК ---

            if not is_last_plot: ax.tick_params(labelbottom=False)
            print(f"      Настройки осей и заголовка для мини-спектрограммы применены.")

        except IndexError:
             print(f"    ❌ ОШИБКА: Индекс {current_ax_idx} вне диапазона осей (всего {len(axes)}).")
        except Exception as e_mini_spec:
             print(f"    ❌ НЕИЗВЕСТНАЯ ОШИБКА в блоке мини-спектрограммы: {e_mini_spec}")
             traceback.print_exc(limit=1)
             try:
                 ax = axes[current_ax_idx]
                 ax.text(0.5, 0.5, f'Ошибка отрисовки:\n{e_mini_spec}', horizontalalignment='center', verticalalignment='center', transform=ax.transAxes, color='red', fontsize=9)
                 current_ax_idx += 1
             except IndexError: pass
    else:
         print("  >>> Пропуск блока отрисовки мини-спектрограммы (данные отсутствуют, пусты или содержат NaN/inf).")

    # --- Следующий блок начнется здесь ---
    # 9. RMS Plot с Адаптивным Порогом
    # ... (остальной код функции) ...

    # 9. RMS Plot с Адаптивным Порогом
    if 'rms_plot' in plot_indices and rms_values is not None:
        ax = axes[current_ax_idx]; current_ax_idx += 1; is_last_plot = (current_ax_idx == num_total_plots)
        if len(rms_values) > n_frames: rms_values_plot = rms_values[:n_frames]
        elif len(rms_values) < n_frames: rms_values_plot = np.pad(rms_values, (0, n_frames - len(rms_values)), mode='edge')
        else: rms_values_plot = rms_values
        if adaptive_threshold is not None:
            if len(adaptive_threshold) > n_frames: adaptive_threshold_plot = adaptive_threshold[:n_frames]
            elif len(adaptive_threshold) < n_frames: adaptive_threshold_plot = np.pad(adaptive_threshold, (0, n_frames - len(adaptive_threshold)), mode='edge')
            else: adaptive_threshold_plot = adaptive_threshold
        else: adaptive_threshold_plot = None
        ax.plot(times, rms_values_plot, label='RMS Energy', color='purple', alpha=0.8, linewidth=1.5)
        if adaptive_threshold_plot is not None and offset_for_title is not None:
             ax.plot(times, adaptive_threshold_plot, color='red', linestyle='--', label=f'Адаптивный порог (Mean + {offset_for_title:.4f})')
             ax.fill_between(times, 0, adaptive_threshold_plot, color='red', alpha=0.1)
        ax.set_title(f"RMS Energy & Адаптивный Порог (Окно: {duration_for_title:.2f}s, Отступ: {offset_for_title:.4f})")
        ax.set_xlabel("Время (с)" if is_last_plot else None); ax.set_ylabel("RMS"); ax.legend(fontsize='small', loc='upper right'); ax.grid(True, linestyle='--', alpha=0.6)
        ax.set_xlim(times[0], times[-1])
        if not is_last_plot: ax.tick_params(labelbottom=False)

    # 10. Маска Сигнал/Шум
    if 'mask_plot' in plot_indices and binary_mask is not None:
        ax = axes[current_ax_idx]; current_ax_idx += 1; is_last_plot = (current_ax_idx == num_total_plots)
        if len(binary_mask) > n_frames: binary_mask_plot = binary_mask[:n_frames]
        elif len(binary_mask) < n_frames: binary_mask_plot = np.pad(binary_mask, (0, n_frames - len(binary_mask)), mode='edge')
        else: binary_mask_plot = binary_mask
        mask_image = binary_mask_plot[np.newaxis, :]
        img = ax.imshow(mask_image, aspect='auto', cmap='gray_r', interpolation='nearest', extent=[times[0], times[-1], 0, 1], vmin=0, vmax=1)
        ax.set_title('Маска Сигнал (1) / Шум (0)')
        ax.set_xlabel("Время (с)" if is_last_plot else None); ax.set_yticks([]); ax.set_ylabel("Маска")
        if not is_last_plot: ax.tick_params(labelbottom=False)

    # 11. SpecAugment Before (dB)
    if 'spec_before_aug' in plot_indices and spec_before_aug is not None:
        ax = axes[current_ax_idx]; current_ax_idx += 1; is_last_plot = (current_ax_idx == num_total_plots)
        spec_db = librosa.amplitude_to_db(spec_before_aug, ref=np.max) if np.min(spec_before_aug) >= 0 else spec_before_aug
        img = librosa.display.specshow(spec_db, sr=sr, hop_length=hop_length, x_axis='time', y_axis='linear', ax=ax, fmax=vis_fmax, cmap=DB_CMAP, n_fft=n_fft)
        ax.set_title(f"{spec_aug_title} (Before SpecAugment, dB)")
        ax.set_xlabel("Время (с)" if is_last_plot else None); ax.set_ylabel("Частота (Hz)"); fig.colorbar(img, ax=ax, format='%+2.0f dB')
        if not is_last_plot: ax.tick_params(labelbottom=False)

    # 12. SpecAugment After (dB)
    if 'spec_after_aug' in plot_indices and spec_after_aug is not None:
        ax = axes[current_ax_idx]; current_ax_idx += 1; is_last_plot = (current_ax_idx == num_total_plots)
        spec_db = librosa.amplitude_to_db(spec_after_aug, ref=np.max) if np.min(spec_after_aug) >= 0 else spec_after_aug
        img = librosa.display.specshow(spec_db, sr=sr, hop_length=hop_length, x_axis='time', y_axis='linear', ax=ax, fmax=vis_fmax, cmap=DB_CMAP, n_fft=n_fft)
        ax.set_title(f"{spec_aug_title} (After SpecAugment, dB)")
        ax.set_xlabel("Время (с)" if is_last_plot else None); ax.set_ylabel("Частота (Hz)"); fig.colorbar(img, ax=ax, format='%+2.0f dB')
        if not is_last_plot: ax.tick_params(labelbottom=False)


    # --- Финальная настройка layout ---
    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.subplots_adjust(hspace=0.30) # Компактный интервал
    plt.show()

print("Функция визуализации (v10.3) определена.")

# --- Основная функция обработки (v10 - Использует новые расчеты и отрисовку) ---
def process_and_plot_combined_v10(
    # Параметры STFT / Mel
    n_fft, hop_length, n_mels,
    # Параметры Адаптивной Маски
    threshold_smoothing_duration_s, threshold_offset, min_absolute_rms_threshold,
    # Параметры Постобработки Маски
    apply_median_filter, median_filter_ms,
    # Параметры Нормализации
    normalization_type, pcen_gain, pcen_bias, pcen_power, pcen_time_constant,
    # Параметры Аудио Аугментации
    use_audio_aug, aug_p, aug_add_noise, noise_min_snr, noise_max_snr, noise_p,
    aug_gain, gain_min_db, gain_max_db, gain_p,
    # Параметры SpecAugment
    use_spec_aug, spec_aug_target, time_mask_param, num_time_masks,
    freq_mask_param, num_freq_masks,
    # Параметры Визуализации
    vis_fmax
):
    """
    Загружает аудио, обрабатывает (v10), вызывает plot_combined_representations_v10.
    """
    global current_audio_data
    y = current_audio_data.get('y'); sr = current_audio_data.get('sr'); file_path = current_audio_data.get('path')
    if y is None or sr is None: print("Аудио не загружено."); return

    title_suffix = f"({file_path.name})"
    y_processed = y.copy(); y_augmented = None

    # --- 1. Аудио Аугментации (код без изменений) ---
    if use_audio_aug and aug_p > 0:
        transforms = [];
        if aug_add_noise and noise_p > 0 and 'noise_files_available' in globals() and noise_files_available: transforms.append(audiomentations.AddBackgroundNoise(sounds_path=BACKGROUND_NOISE_PATH, min_snr_in_db=noise_min_snr, max_snr_in_db=noise_max_snr, p=noise_p))
        if aug_gain and gain_p > 0: transforms.append(audiomentations.Gain(min_gain_in_db=gain_min_db, max_gain_in_db=gain_max_db, p=gain_p))
        # Добавьте сюда другие аудио-аугментации, если они есть в виджетах (Clipping, TimeStretch)
        # Пример:
        # clip_params_current = {"min_percentile_threshold": w_clip_min_perc.value, ...} # Получить значения из виджетов
        # if w_clip_p.value > 0: transforms.append(audiomentations.ClippingDistortion(**clip_params_current))
        # stretch_params_current = {"min_rate": w_stretch_min_rate.value, ...}
        # if w_stretch_p.value > 0: transforms.append(audiomentations.TimeStretch(**stretch_params_current))

        if transforms:
            augmenter = audiomentations.Compose(transforms=transforms, p=aug_p)
            try: y_processed = augmenter(samples=y_processed, sample_rate=sr); y_augmented = y_processed; current_audio_data['y_aug'] = y_augmented; print(f"🎧 Аудио аугментации применены (p={aug_p}).")
            except Exception as e: print(f"❌ Ошибка аудио аугментаций: {e}"); y_processed = y.copy(); y_augmented = None; current_audio_data['y_aug'] = None
        else: print("Нет активных аудио аугментаций."); current_audio_data['y_aug'] = None
    else: current_audio_data['y_aug'] = None

    # --- 2. Расчет ВСЕХ признаков и метрик с помощью новой функции ---
    print("📊 Расчет признаков и метрик (v10)...")
    results = calculate_features_and_metrics_v10(
        y=y_processed, sr=sr,
        n_fft=n_fft, hop_length=hop_length, n_mels=n_mels,
        threshold_smoothing_duration_s=threshold_smoothing_duration_s,
        threshold_offset=threshold_offset,
        min_absolute_rms_threshold=min_absolute_rms_threshold,
        apply_median_filter=apply_median_filter,
        median_filter_ms=median_filter_ms,
        normalization_type=normalization_type,
        pcen_gain=pcen_gain, pcen_bias=pcen_bias, pcen_power=pcen_power,
        pcen_time_constant=pcen_time_constant,
        peak_freq_neighbors=2 # Задаем количество соседей для мини-спектрограммы
    )

    if results is None or results.get('linear_amp_spec') is None:
         print("❌ Не удалось рассчитать основные признаки. Отрисовка невозможна.")
         return

    print("✅ Признаки и метрики рассчитаны.")

    # --- 3. SpecAugment (код без изменений, но использует данные из results) ---
    spec_before_aug = None; spec_after_aug = None; spec_aug_title = ""
    if use_spec_aug and (num_time_masks > 0 or num_freq_masks > 0):
        spec_to_augment = None
        # Выбираем цель на основе типа нормализации и наличия данных в results
        if spec_aug_target == 'pcen_linear' and results.get('pcen_linear_spec') is not None: spec_to_augment = results['pcen_linear_spec']; spec_aug_title = "PCEN Linear"
        elif spec_aug_target == 'zscore_linear' and results.get('zscore_linear_spec') is not None: spec_to_augment = results['zscore_linear_spec']; spec_aug_title = "Z-score Linear"
        elif spec_aug_target == 'linear_amp' and normalization_type == 'None' and results.get('linear_amp_spec') is not None: spec_to_augment = results['linear_amp_spec']; spec_aug_title = "Linear Amplitude"
        else: # Fallback logic
            fallback_applied = False
            if results.get('pcen_linear_spec') is not None: spec_to_augment = results['pcen_linear_spec']; spec_aug_title = "PCEN Linear (fallback)"; fallback_applied = True
            elif results.get('zscore_linear_spec') is not None: spec_to_augment = results['zscore_linear_spec']; spec_aug_title = "Z-score Linear (fallback)"; fallback_applied = True
            elif results.get('linear_amp_spec') is not None: spec_to_augment = results['linear_amp_spec']; spec_aug_title = "Linear Amplitude (fallback)"; fallback_applied = True
            if fallback_applied: print(f"⚠️ Цель '{spec_aug_target}' для SpecAugment недоступна, используется '{spec_aug_title}'.")
            else: print(f"⚠️ SpecAugment не может быть применен: цель '{spec_aug_target}' и все fallback цели недоступны.")

        if spec_to_augment is not None:
            spec_before_aug = spec_to_augment.copy(); spec_tensor = torch.tensor(spec_before_aug).unsqueeze(0)
            try:
                if num_freq_masks > 0 and freq_mask_param > 0: freq_masking = torchaudio.transforms.FrequencyMasking(freq_mask_param=freq_mask_param, iid_masks=True); [spec_tensor := freq_masking(spec_tensor) for _ in range(num_freq_masks)]
                if num_time_masks > 0 and time_mask_param > 0: time_masking = torchaudio.transforms.TimeMasking(time_mask_param=time_mask_param, p=1.0, iid_masks=True); [spec_tensor := time_masking(spec_tensor) for _ in range(num_time_masks)]
                spec_after_aug = spec_tensor.squeeze(0).numpy(); print(f"🎭 SpecAugment применен к '{spec_aug_title}'.")
            except Exception as e_sa: print(f"❌ Ошибка при применении SpecAugment: {e_sa}"); spec_after_aug = spec_before_aug
        else: print("SpecAugment пропущен (нет подходящей цели).")

    # --- 4. Вызов ОБНОВЛЕННОЙ Отрисовки (v10) ---
    print("🎨 Вызов функции отрисовки v10...")
    plot_combined_representations_v10(
        results=results, # Передаем словарь с данными
        y=y, sr=sr, y_aug=y_augmented,
        n_fft=n_fft, hop_length=hop_length, n_mels=n_mels,
        offset_for_title=threshold_offset,
        duration_for_title=threshold_smoothing_duration_s,
        spec_before_aug=spec_before_aug, # Передаем результат SpecAugment
        spec_after_aug=spec_after_aug,
        spec_aug_title=spec_aug_title,
        vis_fmax=vis_fmax, title_suffix=title_suffix,
        peak_freq_neighbors=2 # Передаем количество соседей
    )

print("Основная функция обработки (v10) определена.")

# --- Важно: Обновить вызовы в следующих ячейках ---
# Убедитесь, что в Ячейке 6 (Связывание виджетов) и Ячейке 7 (Отображение)
# используются обновленные имена функций и виджетов (v10), если вы их переименовывали.
# Например, кнопка обновления должна вызывать on_update_button_clicked_v10,
# которая, в свою очередь, вызывает process_and_plot_combined_v10.
# Если вы не переименовывали, просто убедитесь, что код выше заменил старую функцию.

Функция calculate_features_and_metrics_v10 определена.
Функция визуализации (v10.3) определена.
Основная функция обработки (v10) определена.


### Ячейка 5: Создание Интерактивных Виджетов

In [69]:
# Ячейка 5: Создание Интерактивных Виджетов (v9)

style_med = {'description_width': '150px'}
style_long = {'description_width': '180px'}
layout_half = Layout(width='48%')
layout_full = Layout(width='98%')

# --- Виджеты STFT / Mel ---
w_n_fft = widgets.SelectionSlider(options=[256,256+64,  384,384+64, 512, 768, 1024, 2048], value=256, description='n_fft:', style=style_med, continuous_update=False)
w_hop_length = widgets.SelectionSlider(options=[64, 96, 128, 192, 256], value=64, description='hop_length:', style=style_med, continuous_update=False) # Default из Optuna v7
# --- НОВЫЙ ВИДЖЕТ ---
w_n_mels = widgets.IntSlider(min=16, max=128, step=4, value=64, description='n_mels (для Mel Sum):', style=style_med, continuous_update=False)
# --- КОНЕЦ НОВОГО ---
w_vis_fmax = widgets.IntSlider(min=1000, max=8000, step=100, value=4000, description='fmax (визуализация):', style=style_long, continuous_update=False)
# --- ОБНОВЛЕН BOX ---
stft_box = VBox([ HBox([w_n_fft, w_hop_length], layout=layout_full), HBox([w_n_mels, w_vis_fmax], layout=layout_full) ], layout=layout_full)
# --- КОНЕЦ ОБНОВЛЕНИЯ ---

# --- Виджеты Адаптивной Маски RMS ---
# --- ПЕРЕИМЕНОВАН ВИДЖЕТ и ОБНОВЛЕН DEFAULT ---
w_threshold_smoothing_duration = widgets.FloatSlider(
    min=0.05, max=2.0, step=0.05, value=0.65, # Default из Optuna v7
    description='Окно сглаж. (с):', style=style_long, continuous_update=False, readout_format='.2f'
)
# --- КОНЕЦ ИЗМЕНЕНИЯ ---
w_threshold_offset = widgets.FloatSlider(
    min=0.0, max=0.05, step=0.001, value=0.001, # Default из Optuna v7
    description='Отступ от сглаж. ср.:', style=style_long, continuous_update=False, readout_format='.4f'
)
w_min_absolute_rms_threshold = widgets.FloatSlider(
    min=0.0, max=0.05, step=0.0005, value=0.005,
    description='Мин. абс. порог RMS:', style=style_long, continuous_update=False, readout_format='.4f'
)
# --- ОБНОВЛЕН BOX ---
rms_mask_box = VBox([w_threshold_smoothing_duration, w_threshold_offset, w_min_absolute_rms_threshold], layout=layout_full)
# --- КОНЕЦ ОБНОВЛЕНИЯ ---

# --- Виджеты Постобработки Маски ---
w_apply_median_filter = widgets.Checkbox(value=True, description='Применять медианный фильтр к маске', indent=False)
w_median_filter_ms = widgets.IntSlider(min=0, max=100, step=5, value=30, description='Длина фильтра (мс):', style=style_long, continuous_update=False, disabled=False)
def toggle_median_filter_slider(change): w_median_filter_ms.disabled = not change['new']
w_apply_median_filter.observe(toggle_median_filter_slider, names='value')
toggle_median_filter_slider({'new': w_apply_median_filter.value})
postprocess_mask_box = VBox([w_apply_median_filter, w_median_filter_ms], layout=Layout(border='1px solid lightgray', padding='5px', margin='5px 0 5px 0'))

# --- Виджеты Нормализации (включая PCEN) ---
w_normalization_type = widgets.RadioButtons(options=['None', 'Z-score', 'PCEN'], value='PCEN', description='Нормализация Linear Amp:', style=style_med, layout=Layout(margin='10px 0 10px 0'))
# --- ОБНОВЛЕНЫ DEFAULTS из Optuna v7 ---
w_pcen_gain = widgets.FloatSlider(min=0.1, max=2.0, step=0.05, value=2.0, description='PCEN Gain (α):', style=style_med, continuous_update=False, readout_format='.2f', disabled=True)
w_pcen_bias = widgets.FloatSlider(min=0.1, max=20.0, step=0.1, value=9.0, description='PCEN Bias (δ):', style=style_med, continuous_update=False, readout_format='.1f', disabled=True)
w_pcen_power = widgets.FloatSlider(min=0.05, max=1.0, step=0.05, value=0.5, description='PCEN Power (r):', style=style_med, continuous_update=False, readout_format='.2f', disabled=True)
w_pcen_time_constant = widgets.FloatSlider(min=0.01, max=1.0, step=0.01, value=0.44, description='PCEN Time Const (s):', style=style_med, continuous_update=False, readout_format='.2f', disabled=True)
# --- КОНЕЦ ОБНОВЛЕНИЯ DEFAULTS ---
pcen_params_box = VBox([ HBox([w_pcen_gain, w_pcen_bias], layout=layout_full), HBox([w_pcen_power, w_pcen_time_constant], layout=layout_full), ], layout=Layout(border='1px solid lightgray', padding='5px', margin='5px 0 5px 0', display='none'))
def toggle_pcen_widgets_v9(norm_type): show_pcen = (norm_type == 'PCEN'); pcen_params_box.layout.display = 'flex' if show_pcen else 'none'; w_pcen_gain.disabled = not show_pcen; w_pcen_bias.disabled = not show_pcen; w_pcen_power.disabled = not show_pcen; w_pcen_time_constant.disabled = not show_pcen
widgets.interactive_output(toggle_pcen_widgets_v9, {'norm_type': w_normalization_type})
toggle_pcen_widgets_v9(w_normalization_type.value)
normalization_box = VBox([w_normalization_type, pcen_params_box])

# --- Виджеты Аудио Аугментации ---
w_use_audio_aug = widgets.Checkbox(value=False, description='Применять Аудио Аугментации', indent=False)
w_aug_p = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.7, description='Общая вер-ть (p):', style=style_med, continuous_update=False, readout_format='.2f', disabled=True)
w_aug_add_noise = widgets.Checkbox(value=True, description='AddBackgroundNoise', indent=False, disabled=True or not noise_files_available)
w_noise_min_snr = widgets.FloatSlider(min=-5.0, max=30.0, step=0.5, value=3.0, description='Min SNR (dB):', style=style_med, continuous_update=False, readout_format='.1f', disabled=True)
w_noise_max_snr = widgets.FloatSlider(min=0.0, max=40.0, step=0.5, value=15.0, description='Max SNR (dB):', style=style_med, continuous_update=False, readout_format='.1f', disabled=True)
w_noise_p = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.5, description='Вер-ть шума (p):', style=style_med, continuous_update=False, readout_format='.2f', disabled=True)
noise_box = VBox([w_aug_add_noise, HBox([w_noise_min_snr, w_noise_max_snr]), w_noise_p], layout=Layout(border='1px solid lightgray', padding='5px', margin='2px'))
w_aug_gain = widgets.Checkbox(value=True, description='Gain', indent=False, disabled=True)
w_gain_min_db = widgets.FloatSlider(min=-20.0, max=0.0, step=0.5, value=-6.0, description='Min Gain (dB):', style=style_med, continuous_update=False, readout_format='.1f', disabled=True)
w_gain_max_db = widgets.FloatSlider(min=0.0, max=20.0, step=0.5, value=6.0, description='Max Gain (dB):', style=style_med, continuous_update=False, readout_format='.1f', disabled=True)
w_gain_p = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.4, description='Вер-ть Gain (p):', style=style_med, continuous_update=False, readout_format='.2f', disabled=True)
gain_box = VBox([w_aug_gain, HBox([w_gain_min_db, w_gain_max_db]), w_gain_p], layout=Layout(border='1px solid lightgray', padding='5px', margin='2px'))
def toggle_audio_aug_widgets_v9(use_audio_aug):
    disabled = not use_audio_aug; w_aug_p.disabled = disabled;
    w_aug_add_noise.disabled = disabled or not noise_files_available
    noise_disabled = disabled or not w_aug_add_noise.value or not noise_files_available
    w_noise_min_snr.disabled = noise_disabled; w_noise_max_snr.disabled = noise_disabled; w_noise_p.disabled = noise_disabled
    w_aug_gain.disabled = disabled
    gain_disabled = disabled or not w_aug_gain.value
    w_gain_min_db.disabled = gain_disabled; w_gain_max_db.disabled = gain_disabled; w_gain_p.disabled = gain_disabled
widgets.interactive_output(toggle_audio_aug_widgets_v9, {'use_audio_aug': w_use_audio_aug})
widgets.interactive_output(toggle_audio_aug_widgets_v9, {'use_audio_aug': w_aug_add_noise})
widgets.interactive_output(toggle_audio_aug_widgets_v9, {'use_audio_aug': w_aug_gain})
toggle_audio_aug_widgets_v9(w_use_audio_aug.value)
audio_aug_box = VBox([ HBox([w_use_audio_aug, w_aug_p]), noise_box, gain_box, ], layout=layout_full)

# --- Виджеты SpecAugment ---
w_use_spec_aug = widgets.Checkbox(value=False, description='Применять SpecAugment', indent=False)
w_spec_aug_target = widgets.Dropdown(options=['linear_amp', 'zscore_linear', 'pcen_linear'], value='pcen_linear', description='Цель SpecAugment:', style=style_med, disabled=True)
w_num_time_masks = widgets.IntSlider(min=0, max=10, step=1, value=2, description='Кол-во Time масок:', style=style_med, continuous_update=False, disabled=True)
w_time_mask_param = widgets.IntSlider(min=1, max=50, step=1, value=5, description='Ширина Time маски:', style=style_med, continuous_update=False, disabled=True)
w_num_freq_masks = widgets.IntSlider(min=0, max=10, step=1, value=2, description='Кол-во Freq масок:', style=style_med, continuous_update=False, disabled=True)
w_freq_mask_param = widgets.IntSlider(min=1, max=30, step=1, value=8, description='Ширина Freq маски:', style=style_med, continuous_update=False, disabled=True)
def toggle_spec_aug_widgets_v9(use_spec_aug):
    disabled = not use_spec_aug; w_spec_aug_target.disabled = disabled;
    w_num_time_masks.disabled = disabled; w_time_mask_param.disabled = disabled or w_num_time_masks.value == 0
    w_num_freq_masks.disabled = disabled; w_freq_mask_param.disabled = disabled or w_num_freq_masks.value == 0
widgets.interactive_output(toggle_spec_aug_widgets_v9, {'use_spec_aug': w_use_spec_aug})
widgets.interactive_output(toggle_spec_aug_widgets_v9, {'use_spec_aug': w_num_time_masks})
widgets.interactive_output(toggle_spec_aug_widgets_v9, {'use_spec_aug': w_num_freq_masks})
toggle_spec_aug_widgets_v9(w_use_spec_aug.value)
spec_aug_box = VBox([ HBox([w_use_spec_aug, w_spec_aug_target]), HBox([w_num_time_masks, w_time_mask_param]), HBox([w_num_freq_masks, w_freq_mask_param]) ], layout=layout_full)

print("Интерактивные виджеты (v9) созданы.")

Интерактивные виджеты (v9) созданы.


### Ячейка 6: Связывание Виджетов с Функцией Обработки

In [70]:
# Ячейка 6: Связывание Виджетов с Функцией Обработки (v10)
import traceback # Убедимся, что traceback импортирован
import ipywidgets as widgets
from IPython.display import clear_output, display # Убедимся, что display импортирован
from pathlib import Path # Убедимся, что Path импортирован

# --- Убедимся, что виджеты из Ячейки 5 доступны ---
# (Предполагается, что виджеты w_n_fft, w_hop_length, w_n_mels и т.д. определены)
try:
    _ = w_n_fft # Простая проверка существования
except NameError:
    print("❌ ОШИБКА: Виджеты из Ячейки 5 не найдены. Запустите Ячейку 5.")
    # Можно добавить код для создания виджетов здесь как fallback, но лучше запустить Ячейку 5
    raise

# Словарь аргументов для process_and_plot_combined_v10
# Названия ключей должны точно совпадать с аргументами функции
interactive_controls_v10 = {
    # STFT / Mel
    'n_fft': w_n_fft, 'hop_length': w_hop_length, 'n_mels': w_n_mels,
    # Адаптивная Маска RMS
    'threshold_smoothing_duration_s': w_threshold_smoothing_duration,
    'threshold_offset': w_threshold_offset,
    'min_absolute_rms_threshold': w_min_absolute_rms_threshold,
    # Постобработка Маски
    'apply_median_filter': w_apply_median_filter,
    'median_filter_ms': w_median_filter_ms,
    # Normalization Choice
    'normalization_type': w_normalization_type,
    # PCEN Params
    'pcen_gain': w_pcen_gain, 'pcen_bias': w_pcen_bias,
    'pcen_power': w_pcen_power, 'pcen_time_constant': w_pcen_time_constant,
    # Audio Aug
    'use_audio_aug': w_use_audio_aug, 'aug_p': w_aug_p,
    'aug_add_noise': w_aug_add_noise, 'noise_min_snr': w_noise_min_snr, 'noise_max_snr': w_noise_max_snr, 'noise_p': w_noise_p,
    'aug_gain': w_aug_gain, 'gain_min_db': w_gain_min_db, 'gain_max_db': w_gain_max_db, 'gain_p': w_gain_p,
    # SpecAugment
    'use_spec_aug': w_use_spec_aug, 'spec_aug_target': w_spec_aug_target,
    'time_mask_param': w_time_mask_param, 'num_time_masks': w_num_time_masks,
    'freq_mask_param': w_freq_mask_param, 'num_freq_masks': w_num_freq_masks,
    # Visualization
    'vis_fmax': w_vis_fmax,
}

# --- Кнопка Обновления и ее Обработчик (v10) ---
update_button_combined_v10 = widgets.Button(
    description="Обновить Графики (v10)", button_style='info',
    tooltip='Применить текущие настройки и перерисовать все графики v10', icon='refresh',
    layout=Layout(width='220px', margin='20px 0 20px 0')
)
# Используем новую область вывода, чтобы избежать конфликтов, если старая еще используется
plot_output_combined_v10 = widgets.Output()

def on_update_button_clicked_v10(b):
    """Обработчик нажатия кнопки 'Обновить Графики' для v10."""
    with plot_output_combined_v10: # Используем новую область вывода
        clear_output(wait=True)
        print("🔄 Обновление объединенных графиков (v10) с текущими настройками...")
        # Собираем значения из виджетов, определенных в Ячейке 5
        current_widget_values = {}
        missing_widgets = []
        for name, widget in interactive_controls_v10.items():
            if widget is None:
                print(f"Предупреждение: Виджет для '{name}' не найден.")
                missing_widgets.append(name)
                continue
            current_widget_values[name] = widget.value

        if missing_widgets:
            print(f"❌ Ошибка: Отсутствуют виджеты для параметров: {', '.join(missing_widgets)}. Обновление невозможно.")
            return

        # Проверяем наличие функции обработки v10
        if 'process_and_plot_combined_v10' not in globals():
             print("❌ Ошибка: Функция process_and_plot_combined_v10 не определена. Запустите Ячейку 4.")
             return

        try:
            # Вызываем ОБНОВЛЕННУЮ функцию обработки и отрисовки v10
            process_and_plot_combined_v10(**current_widget_values)
            print("✅ Графики обновлены.")
        except Exception as e:
            print(f"❌ Ошибка при обновлении объединенных графиков (v10): {e}")
            traceback.print_exc(limit=2)

update_button_combined_v10.on_click(on_update_button_clicked_v10)

# --- Обработчик Смены Файла (v10) ---
def handle_file_change_v10(change):
    """Перерисовывает графики при смене файла для v10."""
    global plot_output_combined_v10 # Используем новую область вывода

    # Сначала обновляем информацию о файле и загружаем аудио (функция из ячейки 3)
    if 'display_file_info' in globals():
        # Убедимся, что display_file_info существует и вызываем ее
        display_file_info(change['new'])
    else:
        print("Warning: Функция display_file_info не найдена.")
        # Попытка загрузить аудио вручную, если display_file_info недоступна
        try:
             global current_audio_data
             new_path = Path(change['new'])
             if new_path.exists():
                 y, sr = sf.read(new_path, dtype='float32')
                 current_audio_data['y'] = y
                 current_audio_data['sr'] = sr
                 current_audio_data['path'] = new_path
                 current_audio_data['y_aug'] = None
                 print(f"Аудио {new_path.name} загружено вручную.")
             else:
                 print(f"Файл {new_path} не найден.")
                 current_audio_data = {'y': None, 'sr': None, 'path': None, 'y_aug': None}
        except Exception as e_load:
             print(f"Ошибка ручной загрузки аудио: {e_load}")
             current_audio_data = {'y': None, 'sr': None, 'path': None, 'y_aug': None}


    # Затем запускаем полную перерисовку с текущими настройками виджетов
    with plot_output_combined_v10: # Используем новую область вывода
        clear_output(wait=True)
        new_file_path = Path(change['new'])
        print(f"⬇️ Загрузка нового файла: {new_file_path.name} и отрисовка (v10)...")

        # Собираем значения виджетов
        current_widget_values = {}
        missing_widgets = []
        for name, widget in interactive_controls_v10.items():
             if widget is None: missing_widgets.append(name); continue
             current_widget_values[name] = widget.value
        if missing_widgets:
             print(f"❌ Ошибка: Отсутствуют виджеты: {', '.join(missing_widgets)}.")
             return
        if 'process_and_plot_combined_v10' not in globals():
             print("❌ Ошибка: Функция process_and_plot_combined_v10 не определена."); return

        try:
            # Вызываем ОБНОВЛЕННУЮ функцию обработки и отрисовки v10
            process_and_plot_combined_v10(**current_widget_values)
        except Exception as e:
            print(f"❌ Ошибка при обработке нового файла (v10): {e}")
            traceback.print_exc(limit=2)

# Связываем обработчик смены файла с виджетом file_selector (определен в ячейке 3)
try:
    if 'file_selector' in locals() and isinstance(file_selector, widgets.Dropdown):
        # Удаляем старый наблюдатель, если он был
        # file_selector.unobserve_all() # Раскомментировать, если были проблемы с двойным вызовом
        file_selector.observe(handle_file_change_v10, names='value')
        print("Обработчик смены файла (v10) привязан к file_selector.")
    else:
        print("Warning: file_selector не найден или имеет неверный тип. Смена файла не будет вызывать обновление.")
except NameError:
     print("Warning: file_selector не определен.")


print("Кнопка обновления и обработчики событий (v10) настроены.")

Обработчик смены файла (v10) привязан к file_selector.
Кнопка обновления и обработчики событий (v10) настроены.


### Ячейка 7: Отображение Интерфейса

In [71]:
# Ячейка 7: Отображение Интерфейса (v10)
import ipywidgets as widgets
from IPython.display import display, HTML # Убедимся, что HTML импортирован

# --- Убедимся, что виджеты и боксы из Ячейки 5 доступны ---
try:
    _ = stft_box # Простая проверка существования
    _ = update_button_combined_v10 # Проверка кнопки из Ячейки 6
    _ = plot_output_combined_v10 # Проверка области вывода из Ячейки 6
except NameError:
    print("❌ ОШИБКА: Необходимые виджеты или контейнеры из Ячеек 5/6 не найдены. Запустите их.")
    raise

# Собираем все виджеты в Аккордеон (v10)
# Используем контейнеры из Ячейки 5 (stft_box, rms_mask_box и т.д.)
accordion_v10 = widgets.Accordion(children=[
    stft_box,
    rms_mask_box,
    postprocess_mask_box,
    normalization_box,
    audio_aug_box,
    spec_aug_box
])
# Устанавливаем заголовки (можно оставить как есть или обновить до v10)
accordion_v10.set_title(0, '1. Параметры STFT / Mel')
accordion_v10.set_title(1, '2. Адаптивная Маска RMS')
accordion_v10.set_title(2, '3. Постобработка Маски')
accordion_v10.set_title(3, '4. Нормализация')
accordion_v10.set_title(4, '5. Аудио Аугментации')
accordion_v10.set_title(5, '6. SpecAugment')
accordion_v10.selected_index = 0 # Открываем первый раздел по умолчанию

# Собираем весь UI (v10)
# Убедимся, что file_selector и file_info_output из Ячейки 3 доступны
try:
    _ = file_selector
    _ = file_info_output
except NameError:
     print("❌ ОШИБКА: Виджеты file_selector или file_info_output из Ячейки 3 не найдены.")
     # Создаем заглушки, чтобы код не падал, но интерфейс будет неполным
     file_selector = widgets.HTML("<b>Ошибка: file_selector не найден</b>")
     file_info_output = widgets.HTML("<b>Ошибка: file_info_output не найден</b>")

ui_v10 = widgets.VBox([
    widgets.HTML("<h2>Интерактивный Анализатор Аудио Морзе (v10 - с Пиковой Частотой)</h2>"),
    # Виджеты выбора файла и информации о нем (из Ячейки 3)
    file_selector,
    file_info_output,
    widgets.HTML("<hr><h3>Настройки Обработки:</h3>"),
    accordion_v10, # Аккордеон с настройками (v10)
    update_button_combined_v10, # Кнопка обновления (v10)
    widgets.HTML("<hr><h3>Результаты Визуализации:</h3>"),
    plot_output_combined_v10 # Область для графиков (v10)
])

# Отображаем интерфейс
display(ui_v10)

# Выполняем первичную отрисовку при запуске ячейки (v10)
print("\n⚙️ Выполняется первичная отрисовка (v10)...")
# Проверяем наличие функции перед вызовом
if 'on_update_button_clicked_v10' in globals():
    on_update_button_clicked_v10(None) # Вызываем обработчик кнопки v10
    print("\n✅ Интерфейс v10 отображен.")
else:
    print("❌ ОШИБКА: Функция on_update_button_clicked_v10 не найдена. Первичная отрисовка не выполнена.")


⚙️ Выполняется первичная отрисовка (v10)...



✅ Интерфейс v10 отображен.


In [72]:
# Ячейка 2: Конфигурация (v8.8 - Замена AddBackgroundNoise, новые аугментации)
import time
CONFIG = {
    # ==========================================================================
    #                           ОСНОВНЫЕ НАСТРОЙКИ
    # ==========================================================================
    # !!! Обнови описание запуска !!!
    "run_description": "CRNN_hop96_LinearAmpSpec_v8.8_FT_OneCycleLR_NoBgNoise",

    # --- Режим только дообучения ---
    "run_mode": "finetune_only",

    # --- !!! ВАЖНО: Укажи ПРАВИЛЬНЫЙ путь к модели для дообучения !!! ---
    "finetune_only_checkpoint_path": "./output_v8.3_linear_amp_spec/CRNN_hop96_LinearAmpSpec_v8.3/morse_v8.3_linamp_CRNN_hop96_LinearAmpSpec_v8.3_TRAIN_ReduceLROnPlateau_20250418_110720_best_epoch15_best_lev0.4430.pth", # Пример! Замени на свой путь

    # --- Режим работы (Калибровка) ---
    "mode": { "calibration_mode": False, "calibration_subset_size": 500, "calibration_epochs": 5, },
    "random_seed": 42,

    # --- Пути ---
    "paths": {
        "data_dir": "./", "audio_folder_name": "morse_dataset/morse_dataset",
        # !!! Новая папка вывода !!!
        "output_dir": "./output_v8.8_FT_OneCycleLR_NoBgNoise",
        "mlflow_tracking_uri": None,
        # background_noise_dir больше не используется напрямую здесь, но оставим путь
        "background_noise_dir": "./_background_noise_",
    },

    # --- MLflow Логирование ---
    # !!! Обнови имя эксперимента и префикс !!!
    "mlflow": { "experiment_name": "Morse Code Recognition v8.8 FT NoBgNoise", "run_name_prefix": "morse_v8.8_ft_ocl_nbn", },

    # ==========================================================================
    #                           ПАРАМЕТРЫ ДАННЫХ
    # ==========================================================================
    "audio": {
        "sample_rate": 8000, "n_fft": 512, "hop_length": 96,
        "use_deltas": False,
    },
    # --- Аугментации Аудио ---
    "audio_augmentation": {
        "apply": True,
        # Можно оставить 0.9 или чуть уменьшить, т.к. pipeline стал другим
        "p": 0.85,
        "pipeline": [
            # --- !!! УДАЛЕНО: AddBackgroundNoise !!! ---
            # {"name": "AddBackgroundNoise", ...}

            # --- Усиленный Гауссов шум ---
            {
                "name": "AddGaussianNoise",
                "params": {
                    "min_amplitude": 0.001,
                    # Можно немного увеличить верхнюю границу
                    "max_amplitude": 0.015,
                    "p": 0.6 # Увеличили вероятность
                }
            },
            # --- Более частое изменение громкости ---
            {
                "name": "Gain",
                "params": {
                    "min_gain_db": -6.0,
                    "max_gain_db": 6.0,
                    "p": 0.7 # Увеличили вероятность
                }
            },
            # --- !!! ДОБАВЛЕНО: Искажение клиппингом !!! ---
            {
                "name": "ClippingDistortion",
                "params": {
                    "min_percentile_threshold": 0, # Начинаем клиппинг с самых тихих
                    "max_percentile_threshold": 10,# Клиппим до 10% самых громких
                    "p": 0.3 # Применяем не слишком часто
                }
            },
            # --- !!! ДОБАВЛЕНО: Растяжение/сжатие времени !!! ---
            {
                "name": "TimeStretch",
                "params": {
                    "min_rate": 0.9, # Замедление до 10%
                    "max_rate": 1.1, # Ускорение до 10%
                    "p": 0.3 # Применяем не слишком часто
                }
            },
            # --- !!! ОПЦИОНАЛЬНО: Сжатие MP3 (требует lameenc) !!! ---
            # {
            #     "name": "Mp3Compression",
            #     "params": {
            #         "min_bitrate": 32,
            #         "max_bitrate": 128, # Довольно сильное сжатие
            #         "backend": "lameenc", # Убедись, что lameenc установлен (pip install lameenc)
            #         "p": 0.2
            #     }
            # }
        ]
    },
    # --- SpecAugment (оставляем без изменений) ---
    "spec_augmentation": {
        "apply": True, "time_mask_param": 10, "num_time_masks": 4,
        "freq_mask_param": 8, "num_freq_masks": 4, "iid_masks": True,
    },

    # ==========================================================================
    #                           ПАРАМЕТРЫ МОДЕЛИ (CRNN)
    # ==========================================================================
    "model": {
        "cnn_out_channels": [32, 64], "cnn_kernel_size": [[3, 3], [3, 3]],
        "cnn_stride": [[2, 2], [2, 2]],
        "rnn_hidden_size": 256, "rnn_layers": 2, "rnn_dropout": 0.2,
        "dropout": 0.2,
        # "vocab_size": ..., "freq_dim": ..., добавляются ниже
    },

    # ==========================================================================
    #                   ПАРАМЕТРЫ ОБУЧЕНИЯ И ОПТИМИЗАЦИИ
    # ==========================================================================
    "training": { # Параметры для основного обучения (если бы оно запускалось)
        "apply_audio_augmentation": False, "epochs": 50,
        "batch_size": 8,
        "optimizer": "AdamW", "learning_rate": 3e-4, "weight_decay": 1e-4,
        "scheduler": "ReduceLROnPlateau",
        "use_amp": True,
        "gradient_accumulation_steps": 2,
        "gradient_clip_val": 1.0, "val_split_ratio": 0.1, "early_stopping_patience": 13,
    },
    "finetuning": { # Параметры для дообучения
        "apply": True,
        "apply_audio_augmentation": True,
        "epochs": 30,
        "batch_size": 8,
        "learning_rate": 2e-4, # max_lr для OneCycleLR
        "scheduler": "OneCycleLR",
        "early_stopping_patience": 13,
        "weight_decay": 1e-5,
        "use_amp": True,
        "gradient_accumulation_steps": 2,
        "gradient_clip_val": 1.0,
    },
    "schedulers": {
         "ReduceLROnPlateau": {
            "mode": "min", "factor_train": 0.5, "factor_finetune": 0.5,
            "patience_train": 4, "patience_finetune": 4, "min_lr": 1e-7, "verbose": True,
        },
         "OneCycleLR": {
            "pct_start": 0.3, "anneal_strategy": "cos",
            "div_factor": 25.0, "final_div_factor": 10000.0,
            "verbose": False
         }
    },
    "ctc": { "blank_char": "<blank>", "blank_idx": 0, "pad_char": "<pad>", "pad_idx": -1, },
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    # --- !!! Оставляем увеличенное значение !!! ---
    "num_workers": 0,
}

# --- Обновление пути к шумам ---
# Этот блок больше не нужен для AddBackgroundNoise, но оставим его на случай,
# если другие аугментации потребуют внешних файлов в будущем.
# try:
#     noise_dir_path = CONFIG["paths"]["background_noise_dir"]
#     if not Path(noise_dir_path).is_dir():
#          print(f"!!! ПРЕДУПРЕЖДЕНИЕ: Папка '{noise_dir_path}' не найдена!")
#          noise_dir_path_resolved = None
#     else:
#          noise_dir_path_resolved = str(Path(noise_dir_path).resolve())
#     # Пример установки пути для гипотетической будущей аугментации
#     # for aug_config in CONFIG.get("audio_augmentation", {}).get("pipeline", []):
#     #     if aug_config.get("name") == "SomeFutureAugmentation":
#     #         if noise_dir_path_resolved:
#     #              aug_config["params"]["external_files_path"] = noise_dir_path_resolved
#     #              print(f"Путь для SomeFutureAugmentation установлен: {noise_dir_path_resolved}")
#     #         else:
#     #              aug_config["params"]["external_files_path"] = None
#     #              print(f"Путь для SomeFutureAugmentation НЕ установлен.")
#     #         break
# except Exception as e:
#     print(f"Предупреждение при установке путей для аугментаций: {e}")
print("Блок обновления пути к шумам для AddBackgroundNoise больше не активен.")


# --- Применение настроек калибровки ---
if CONFIG["mode"]["calibration_mode"]:
    CONFIG["training"]["batch_size"] = 16
    CONFIG["finetuning"]["batch_size"] = 16
    CONFIG["training"]["gradient_accumulation_steps"] = 1
    CONFIG["finetuning"]["gradient_accumulation_steps"] = 1
    print("!!! РЕЖИМ КАЛИБРОВКИ АКТИВЕН !!!")

# --- Добавляем РАЗМЕРНОСТЬ ЧАСТОТ STFT в параметры модели ---
CONFIG["model"]["freq_dim"] = CONFIG["audio"]["n_fft"] // 2 + 1
print(f"!!! Используется Линейная Амплитудная Спектрограмма. PCEN и Log отключены. !!!")
print(f"!!! Размерность частот (freq_dim): {CONFIG['model']['freq_dim']} !!!")
print(f"!!! Batch_size: {CONFIG['training']['batch_size']} (train) / {CONFIG['finetuning']['batch_size']} (FT) !!!")
print(f"!!! Gradient_accumulation_steps: {CONFIG['training']['gradient_accumulation_steps']} (train) / {CONFIG['finetuning']['gradient_accumulation_steps']} (FT) !!!")
print(f"!!! РЕЖИМ ЗАПУСКА: {CONFIG['run_mode']} !!!")
print(f"!!! ШЕДУЛЕР ДЛЯ ДООБУЧЕНИЯ: {CONFIG['finetuning']['scheduler']} !!!")
print(f"!!! NUM_WORKERS: {CONFIG['num_workers']} !!!") # Выводим num_workers


# --- Создание выходной директории ---
OUTPUT_DIR = Path(CONFIG["paths"]["output_dir"]) / CONFIG["run_description"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Выходная директория: {OUTPUT_DIR.resolve()}")

# --- Сохранение начального конфига ---
config_save_path = OUTPUT_DIR / "config_initial.json"
# with open(config_save_path, 'w') as f:
#     json.dump(CONFIG, f, indent=4, ensure_ascii=False, default=str)
print(f"Начальная конфигурация сохранена в: {config_save_path}")
print(f"Устройство: {CONFIG['device']}")

# --- Функция создания аудио-аугментатора ---
# Функция create_audio_augmenter теперь не нуждается в специальной обработке
# AddBackgroundNoise, но проверки для Gain и обработка ошибок остаются полезными.
def create_audio_augmenter(config: dict) -> Optional[audiomentations.Compose]:
    aug_config = config.get("audio_augmentation", {})
    if not aug_config.get("apply", False):
        print("Аудио-аугментации отключены глобально ('apply': False).")
        return None

    transforms = []
    pipeline_config = aug_config.get("pipeline", [])
    if not pipeline_config:
        print("Конвейер аудио-аугментаций пуст ('pipeline': []).")
        return None

    print("\n--- Создание конвейера Аудио-Аугментаций ---")
    for item in pipeline_config:
        name = item.get("name")
        params = item.get("params", {}).copy() # Копируем параметры

        if not name:
            print("Предупреждение: Пропущена аугментация без имени в pipeline.")
            continue

        try:
            # --- Автоматическая коррекция имен параметров (оставляем для Gain) ---
            if name == "Gain":
                if "min_gain_in_db" in params:
                    params["min_gain_db"] = params.pop("min_gain_in_db")
                    print("Предупреждение: Параметр 'min_gain_in_db' переименован в 'min_gain_db' для Gain.")
                if "max_gain_in_db" in params:
                    params["max_gain_db"] = params.pop("max_gain_in_db")
                    print("Предупреждение: Параметр 'max_gain_in_db' переименован в 'max_gain_db' для Gain.")

            # --- Проверка для Mp3Compression (если раскомментируешь) ---
            # if name == "Mp3Compression":
            #     try:
            #         import lameenc
            #     except ImportError:
            #         print("!!! ПРЕДУПРЕЖДЕНИЕ: Библиотека 'lameenc' не найдена. Mp3Compression будет пропущена.")
            #         print("    Установите ее: pip install lameenc")
            #         continue # Пропускаем эту аугментацию

            # --- Инициализация аугментации ---
            transform_class = getattr(audiomentations, name)
            transforms.append(transform_class(**params))
            print(f"  + Аугментация: {name} (p={params.get('p', 1.0)})")

        except AttributeError:
             print(f"!!! ОШИБКА: Аугментация с именем '{name}' не найдена в audiomentations!")
             raise ValueError(f"Неизвестное имя аугментации: {name}")
        except TypeError as te:
            print(f"!!! ОШИБКА ИНИЦИАЛИЗАЦИИ {name}: {te}")
            print(f"    Проверьте параметры для {name} в CONFIG: {item.get('params', {})}")
            raise ValueError(f"Не удалось инициализировать аугментацию {name} из-за ошибки в параметрах.") from te
        except Exception as e:
            print(f"Предупреждение: Общая ошибка инициализации {name}: {e}")

    if not transforms:
        print("Конвейер аудио-аугментаций пуст или все аугментации пропущены.")
        return None

    augmenter = audiomentations.Compose(transforms=transforms, p=aug_config.get("p", 1.0))
    print(f"Конвейер аудио-аугментаций создан (общая p={aug_config.get('p', 1.0)}).")
    print("-" * 46)
    return augmenter

# --- Создаем объект аудио-аугментатора ---
try:
    AUDIO_AUGMENTER_GLOBAL = create_audio_augmenter(CONFIG)
    if not AUDIO_AUGMENTER_GLOBAL:
        print("Аудио-аугментации глобально НЕ будут применяться.")
except ValueError as ve:
    print(f"\n❌ КРИТИЧЕСКАЯ ОШИБКА: Не удалось создать аудио-аугментатор: {ve}")
    AUDIO_AUGMENTER_GLOBAL = None
    print("!!! ВНИМАНИЕ: Аудио-аугментации будут ОТКЛЮЧЕНЫ из-за ошибки конфигурации !!!")
    # raise ve
# Ячейка 15: Интерактивная Песочница для Аудио-Аугментаций (v1.3 - Добавлен клиппинг для normalize=False)
# Позволяет выбрать файл, настроить параметры и послушать результат

import ipywidgets as widgets
from IPython.display import display, Audio, clear_output
import soundfile as sf
import librosa
import numpy as np # Убедимся, что numpy импортирован
import matplotlib.pyplot as plt
import audiomentations
from pathlib import Path
import random
import traceback
import time

# --- Получаем пути и параметры из основного CONFIG ---
try:
    if 'CONFIG' not in globals(): raise NameError("Словарь CONFIG не найден. Запустите ячейку 2.")
    data_dir = Path(CONFIG["paths"]["data_dir"])
    audio_folder_path = data_dir / CONFIG["paths"]["audio_folder_name"]
    target_sr = CONFIG["audio"]["sample_rate"]
    current_audio_aug_config = CONFIG.get("audio_augmentation", {})
    current_pipeline = current_audio_aug_config.get("pipeline", [])
    def get_current_params(name, default_params):
        for item in current_pipeline:
            if item.get("name") == name: return item.get("params", default_params).copy()
        return default_params.copy()
    gauss_params = get_current_params("AddGaussianNoise", {"min_amplitude": 0.001, "max_amplitude": 0.015, "p": 0.6})
    gain_params = get_current_params("Gain", {"min_gain_db": -6.0, "max_gain_db": 6.0, "p": 0.7})
    clip_params = get_current_params("ClippingDistortion", {"min_percentile_threshold": 0, "max_percentile_threshold": 10, "p": 0.3})
    stretch_params = get_current_params("TimeStretch", {"min_rate": 0.9, "max_rate": 1.1, "p": 0.3})
except NameError as e:
    print(f"Ошибка: {e}. Не удалось получить параметры из CONFIG.")
    audio_folder_path = Path("./morse_dataset/morse_dataset") # Пример
    target_sr = 8000
    gauss_params = {"min_amplitude": 0.001, "max_amplitude": 0.015, "p": 0.6}
    gain_params = {"min_gain_db": -6.0, "max_gain_db": 6.0, "p": 0.7}
    clip_params = {"min_percentile_threshold": 0, "max_percentile_threshold": 10, "p": 0.3}
    stretch_params = {"min_rate": 0.9, "max_rate": 1.1, "p": 0.3}

# --- Виджеты для выбора файла ---
print(f"Поиск аудиофайлов в: {audio_folder_path}")
try:
    example_files = sorted([f.name for f in audio_folder_path.glob('*.opus')])[:100]
    if not example_files:
        print(f"!!! Предупреждение: Не найдено .opus файлов в {audio_folder_path}")
        example_files = ["Файлы не найдены"]
except Exception as e:
    print(f"Ошибка при поиске файлов: {e}")
    example_files = ["Ошибка поиска"]
file_selector = widgets.Dropdown(options=example_files, description='Аудиофайл:', style={'description_width': 'initial'}, layout={'width': 'max-content'})

# --- Виджеты для параметров аугментаций ---
# (Код виджетов без изменений)
gauss_p = widgets.FloatSlider(value=gauss_params.get('p', 0.6), min=0.0, max=1.0, step=0.05, description='P (Gauss):', readout_format='.2f')
gauss_min_amp = widgets.FloatLogSlider(value=gauss_params.get('min_amplitude', 0.001), base=10, min=-4, max=-1, step=0.1, description='Min Amp:', readout_format='.4f')
gauss_max_amp = widgets.FloatLogSlider(value=gauss_params.get('max_amplitude', 0.015), base=10, min=-3, max=-0.5, step=0.1, description='Max Amp:', readout_format='.4f')
gauss_box = widgets.VBox([widgets.Label("AddGaussianNoise"), gauss_p, gauss_min_amp, gauss_max_amp])
gain_p = widgets.FloatSlider(value=gain_params.get('p', 0.7), min=0.0, max=1.0, step=0.05, description='P (Gain):', readout_format='.2f')
gain_min_db = widgets.FloatSlider(value=gain_params.get('min_gain_db', -6.0), min=-18.0, max=0.0, step=0.5, description='Min Gain (dB):')
gain_max_db = widgets.FloatSlider(value=gain_params.get('max_gain_db', 6.0), min=0.0, max=18.0, step=0.5, description='Max Gain (dB):')
gain_box = widgets.VBox([widgets.Label("Gain"), gain_p, gain_min_db, gain_max_db])
clip_p = widgets.FloatSlider(value=clip_params.get('p', 0.3), min=0.0, max=1.0, step=0.05, description='P (Clip):', readout_format='.2f')
clip_min_perc = widgets.IntSlider(value=clip_params.get('min_percentile_threshold', 0), min=0, max=50, step=1, description='Min Perc:')
clip_max_perc = widgets.IntSlider(value=clip_params.get('max_percentile_threshold', 10), min=0, max=50, step=1, description='Max Perc:')
clip_box = widgets.VBox([widgets.Label("ClippingDistortion"), clip_p, clip_min_perc, clip_max_perc])
stretch_p = widgets.FloatSlider(value=stretch_params.get('p', 0.3), min=0.0, max=1.0, step=0.05, description='P (Stretch):', readout_format='.2f')
stretch_min_rate = widgets.FloatSlider(value=stretch_params.get('min_rate', 0.9), min=0.7, max=1.0, step=0.01, description='Min Rate:')
stretch_max_rate = widgets.FloatSlider(value=stretch_params.get('max_rate', 1.1), min=1.0, max=1.3, step=0.01, description='Max Rate:')
stretch_box = widgets.VBox([widgets.Label("TimeStretch"), stretch_p, stretch_min_rate, stretch_max_rate])

# --- Кнопка и область вывода ---
apply_button = widgets.Button(description="Применить и Послушать")
output_area = widgets.Output()

# --- Функция-обработчик нажатия кнопки ---
def on_apply_button_clicked(b):
    with output_area:
        clear_output(wait=True)
        file_id = file_selector.value
        if not file_id or file_id in ["Файлы не найдены", "Ошибка поиска"]:
            print("Пожалуйста, выберите корректный аудиофайл.")
            return

        file_path = audio_folder_path / file_id
        print(f"Обработка файла: {file_path}")

        try:
            # 1. Загрузка и ресемплинг ОРИГИНАЛЬНОГО аудио
            y_orig, sr_orig = sf.read(file_path, dtype='float32')
            if sr_orig != target_sr:
                y_orig = librosa.resample(y=y_orig, orig_sr=sr_orig, target_sr=target_sr)
            print(f"Оригинал загружен. Длина: {len(y_orig)/target_sr:.2f} сек, SR: {target_sr}")

            # --- !!! ИЗМЕНЕНО: Клиппинг перед отображением оригинала !!! ---
            y_orig_clipped = np.clip(y_orig, -1.0, 1.0)
            if np.any(y_orig != y_orig_clipped):
                 print("Предупреждение: Оригинальный сигнал был клиппирован для отображения (выходил за [-1, 1]).")
            display(Audio(y_orig_clipped, rate=target_sr, normalize=False))

            # 2. Создание ДИНАМИЧЕСКОГО конвейера аугментаций
            transforms_dynamic = []
            # (Код сборки transforms_dynamic без изменений)
            gauss_params_current = {"min_amplitude": gauss_min_amp.value, "max_amplitude": gauss_max_amp.value, "p": gauss_p.value}
            if gauss_params_current["p"] > 0: transforms_dynamic.append(audiomentations.AddGaussianNoise(**gauss_params_current))
            gain_params_current = {"min_gain_db": gain_min_db.value, "max_gain_db": gain_max_db.value, "p": gain_p.value}
            if gain_params_current["p"] > 0: transforms_dynamic.append(audiomentations.Gain(**gain_params_current))
            clip_params_current = {"min_percentile_threshold": clip_min_perc.value, "max_percentile_threshold": clip_max_perc.value, "p": clip_p.value}
            if clip_params_current["min_percentile_threshold"] > clip_params_current["max_percentile_threshold"]:
                print("Предупреждение: Min Perc > Max Perc для ClippingDistortion. Меняю их местами.")
                clip_params_current["min_percentile_threshold"], clip_params_current["max_percentile_threshold"] = clip_params_current["max_percentile_threshold"], clip_params_current["min_percentile_threshold"]
            if clip_params_current["p"] > 0: transforms_dynamic.append(audiomentations.ClippingDistortion(**clip_params_current))
            stretch_params_current = {"min_rate": stretch_min_rate.value, "max_rate": stretch_max_rate.value, "p": stretch_p.value}
            if stretch_params_current["min_rate"] > stretch_params_current["max_rate"]:
                 print("Предупреждение: Min Rate > Max Rate для TimeStretch. Меняю их местами.")
                 stretch_params_current["min_rate"], stretch_params_current["max_rate"] = stretch_params_current["max_rate"], stretch_params_current["min_rate"]
            if stretch_params_current["p"] > 0: transforms_dynamic.append(audiomentations.TimeStretch(**stretch_params_current))

            # 3. Применение аугментаций
            if not transforms_dynamic:
                print("\nНет активных аугментаций для применения (все вероятности p=0?).")
                y_aug = y_orig # Используем оригинал, если нет аугментаций
            else:
                augmenter_dynamic = audiomentations.Compose(transforms=transforms_dynamic, p=1.0)
                print("\nПрименение аугментаций с параметрами:")
                for t in augmenter_dynamic.transforms:
                    params_to_show = {k: v for k, v in t.__dict__.items() if not k.startswith('_') and k != 'supports_multichannel'}
                    print(f"  - {t.__class__.__name__}: {params_to_show}")

                start_time = time.time()
                # Применяем аугментации к ОРИГИНАЛЬНОМУ y_orig
                y_aug = augmenter_dynamic(samples=y_orig.copy(), sample_rate=target_sr) # Используем .copy() на всякий случай
                end_time = time.time()
                print(f"Аугментация применена за {end_time - start_time:.3f} сек.")

            print(f"Аугментированное аудио. Длина: {len(y_aug)/target_sr:.2f} сек")
            # --- !!! ИЗМЕНЕНО: Клиппинг перед отображением аугментированного аудио !!! ---
            y_aug_clipped = np.clip(y_aug, -1.0, 1.0)
            if np.any(y_aug != y_aug_clipped):
                 print("Предупреждение: Аугментированный сигнал был клиппирован для отображения (выходил за [-1, 1]).")
            display(Audio(y_aug_clipped, rate=target_sr, normalize=False)) # Используем клиппированный массив

            # 4. Визуализация (используем НЕ клиппированные данные для графиков, чтобы видеть реальный эффект)
            try:
                plt.figure(figsize=(15, 8))
                plt.subplot(2, 1, 1)
                time_axis_orig = np.linspace(0, len(y_orig) / target_sr, num=len(y_orig))
                time_axis_aug = np.linspace(0, len(y_aug) / target_sr, num=len(y_aug)) # Используем y_aug
                plt.plot(time_axis_orig, y_orig, label='Оригинал', alpha=0.7)
                plt.plot(time_axis_aug, y_aug, label='Аугментированный (до клиппинга плеера)', alpha=0.7) # Используем y_aug
                plt.title('Сравнение Амплитуды')
                plt.xlabel('Время (с)')
                plt.ylabel('Амплитуда')
                plt.legend()
                plt.grid(True)
                # Лимиты по Y на основе НЕ клиппированных данных
                max_abs_val = max(np.max(np.abs(y_orig)), np.max(np.abs(y_aug)))
                plt.ylim(-max_abs_val * 1.1, max_abs_val * 1.1)

                plt.subplot(2, 1, 2)
                S_orig = np.abs(librosa.stft(y=y_orig, n_fft=512, hop_length=96))
                S_aug = np.abs(librosa.stft(y=y_aug, n_fft=512, hop_length=96)) # Используем y_aug
                S_dB_orig = librosa.amplitude_to_db(S_orig, ref=np.max)
                S_dB_aug = librosa.amplitude_to_db(S_aug, ref=np.max)
                min_db = min(np.min(S_dB_orig), np.min(S_dB_aug))
                max_db = max(np.max(S_dB_orig), np.max(S_dB_aug))

                plt.figure(figsize=(15, 6))
                plt.subplot(1, 2, 1)
                librosa.display.specshow(S_dB_orig, sr=target_sr, hop_length=96, x_axis='time', y_axis='linear', vmin=min_db, vmax=max_db)
                plt.colorbar(format='%+2.0f dB')
                plt.title('Оригинал (Linear Spec)')

                plt.subplot(1, 2, 2)
                librosa.display.specshow(S_dB_aug, sr=target_sr, hop_length=96, x_axis='time', y_axis='linear', vmin=min_db, vmax=max_db)
                plt.colorbar(format='%+2.0f dB')
                plt.title('Аугментированный (Linear Spec)')

                plt.tight_layout()
                plt.show()

            except Exception as plot_e:
                print(f"Не удалось построить графики: {plot_e}")


        except FileNotFoundError:
            print(f"Ошибка: Файл не найден - {file_path}")
        except Exception as e:
            print(f"Произошла ошибка при обработке файла или аугментации:")
            traceback.print_exc()

# --- Привязка обработчика к кнопке ---
apply_button.on_click(on_apply_button_clicked)

# --- Отображение виджетов ---
controls_layout = widgets.VBox([
    file_selector,
    widgets.HBox([gauss_box, gain_box]),
    widgets.HBox([clip_box, stretch_box]),
    apply_button
])

display(controls_layout, output_area)

Блок обновления пути к шумам для AddBackgroundNoise больше не активен.
!!! Используется Линейная Амплитудная Спектрограмма. PCEN и Log отключены. !!!
!!! Размерность частот (freq_dim): 257 !!!
!!! Batch_size: 8 (train) / 8 (FT) !!!
!!! Gradient_accumulation_steps: 2 (train) / 2 (FT) !!!
!!! РЕЖИМ ЗАПУСКА: finetune_only !!!
!!! ШЕДУЛЕР ДЛЯ ДООБУЧЕНИЯ: OneCycleLR !!!
!!! NUM_WORKERS: 0 !!!
Выходная директория: C:\Users\vasja\OneDrive\Рабочий стол\Morse_dev\MorseAudioDecoder\output_v8.8_FT_OneCycleLR_NoBgNoise\CRNN_hop96_LinearAmpSpec_v8.8_FT_OneCycleLR_NoBgNoise
Начальная конфигурация сохранена в: output_v8.8_FT_OneCycleLR_NoBgNoise\CRNN_hop96_LinearAmpSpec_v8.8_FT_OneCycleLR_NoBgNoise\config_initial.json
Устройство: cuda

--- Создание конвейера Аудио-Аугментаций ---
  + Аугментация: AddGaussianNoise (p=0.6)
  + Аугментация: Gain (p=0.7)
  + Аугментация: ClippingDistortion (p=0.3)
  + Аугментация: TimeStretch (p=0.3)
Конвейер аудио-аугментаций создан (общая p=0.85).
---------------------

Output()

In [73]:
# Ячейка для запуска оптимизации Optuna (v11 - Добавлена Log нормализация, фикс. маска)

import optuna
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path
from tqdm.notebook import tqdm
import traceback
import warnings
import pandas as pd
from typing import Tuple, Optional, Dict, List
import json
import math
from scipy.signal import medfilt

# --- Константы и Настройки ---
SAMPLE_RATE = 8000
AUDIO_SUBSET_SIZE = 50
N_TRIALS = 200 # Можно снова увеличить, т.к. добавили вариант
PCEN_DEFAULT_EPS = 1e-6
ZSCORE_EPS = 1e-8
LOG_EPSILON = 1e-8 # Эпсилон для логарифма
ACTIVE_BIN_THRESHOLD_FACTOR = 0.05
FAILED_FILE_PENALTY_FACTOR = 0.2

# --- ФИКСИРОВАННЫЕ Параметры ---
FIXED_N_FFT = 512
FIXED_THRESHOLD_SMOOTHING_DURATION_S = 1.40
FIXED_THRESHOLD_OFFSET = 0.0400
FIXED_MIN_ABSOLUTE_RMS_THRESHOLD = 0.0500
FIXED_APPLY_MEDIAN_FILTER = False
FIXED_MEDIAN_FILTER_MS = 0
# --- КОНЕЦ Фиксированных ---

# --- Пути ---
DATA_DIR = Path("./")
AUDIO_FOLDER_NAME = "morse_dataset/morse_dataset"
TRAIN_CSV_PATH = DATA_DIR / "train.csv"
audio_folder_path = DATA_DIR / AUDIO_FOLDER_NAME

# --- Вспомогательные Функции ---

# Используем функцию v9 для расчета маски (код без изменений)
def calculate_adaptive_mask_mean_v9(rms_values, sr, hop_length,
                                      threshold_smoothing_duration_s: float,
                                      threshold_offset: float,
                                      apply_median_filter: bool = True,
                                      median_filter_ms: int = 30,
                                      min_absolute_rms_threshold: float = 0.005):
    if rms_values is None or rms_values.size == 0: return None, None
    moving_avg_len_frames = int(threshold_smoothing_duration_s * sr / hop_length)
    if moving_avg_len_frames % 2 == 0: moving_avg_len_frames += 1
    if moving_avg_len_frames <= 0: moving_avg_len_frames = 1
    try:
        rms_series = pd.Series(rms_values)
        local_mean_series = rms_series.rolling(window=moving_avg_len_frames, center=True, min_periods=1).mean()
        local_mean = local_mean_series.values
        if np.isnan(local_mean).any() or np.isinf(local_mean).any():
            global_mean = np.nanmean(rms_values); local_mean = np.nan_to_num(local_mean, nan=global_mean if np.isfinite(global_mean) else 0.0)
    except Exception as e_pd:
        global_mean = np.mean(rms_values); local_mean = np.full_like(rms_values, global_mean if np.isfinite(global_mean) else 0.0)
    adaptive_threshold_base = local_mean + threshold_offset
    adaptive_threshold = np.maximum(adaptive_threshold_base, min_absolute_rms_threshold)
    binary_mask_raw = (rms_values >= adaptive_threshold).astype(int)
    if apply_median_filter and median_filter_ms > 0:
        filter_length_frames = int(median_filter_ms * sr / (1000 * hop_length))
        if filter_length_frames % 2 == 0: filter_length_frames += 1
        if filter_length_frames <= 0: filter_length_frames = 1
        if filter_length_frames >= len(binary_mask_raw): binary_mask_filtered = binary_mask_raw
        else:
            try: binary_mask_filtered = medfilt(binary_mask_raw, kernel_size=filter_length_frames)
            except Exception as e_filt: binary_mask_filtered = binary_mask_raw
    else: binary_mask_filtered = binary_mask_raw
    binary_mask = binary_mask_filtered.astype(int)
    if not np.all(np.isfinite(adaptive_threshold)) or not np.all(np.isfinite(binary_mask)):
        adaptive_threshold_base = local_mean + threshold_offset; adaptive_threshold = np.maximum(adaptive_threshold_base, min_absolute_rms_threshold)
        adaptive_threshold = np.nan_to_num(adaptive_threshold, nan=min_absolute_rms_threshold)
        binary_mask = (rms_values >= adaptive_threshold).astype(int); binary_mask = np.nan_to_num(binary_mask, nan=0).astype(int)
        if not np.all(np.isfinite(adaptive_threshold)): return None, None
    return adaptive_threshold, binary_mask


# Функция расчета признаков v11 (добавлена Log нормализация)
def calculate_features_v11_optuna(y: np.ndarray, sr: int, params: dict) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    """
    Рассчитывает признаки с ФИКСИРОВАННЫМИ n_fft и параметрами маски (v11).
    Оптимизируются hop_length, normalization_type ('None', 'Z-score', 'PCEN', 'Log'), PCEN params.
    """
    processed_spectrogram = None
    binary_mask = None
    try:
        # --- УБРАНА ИСКУССТВЕННАЯ ОШИБКА ---
        # if np.random.rand() < 0.1:
        #     return None, None
        # -----------------------------------

        n_fft = FIXED_N_FFT
        hop_length = params['hop_length']
        normalization_type = params['normalization_type']

        # 1. Расчет RMS
        rms_values = librosa.feature.rms(y=y, frame_length=n_fft, hop_length=hop_length)[0]
        if rms_values is None or rms_values.size == 0: return None, None

        # 2. Расчет АДАПТИВНОЙ маски с ФИКСИРОВАННЫМИ параметрами
        _, binary_mask = calculate_adaptive_mask_mean_v9(
            rms_values, sr, hop_length,
            threshold_smoothing_duration_s=FIXED_THRESHOLD_SMOOTHING_DURATION_S,
            threshold_offset=FIXED_THRESHOLD_OFFSET,
            apply_median_filter=FIXED_APPLY_MEDIAN_FILTER,
            median_filter_ms=FIXED_MEDIAN_FILTER_MS,
            min_absolute_rms_threshold=FIXED_MIN_ABSOLUTE_RMS_THRESHOLD
        )
        if binary_mask is None: return None, None

        # 3. Расчет STFT и Linear Amplitude Spectrogram
        S_complex = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
        linear_amp_spec = np.abs(S_complex)

        # 4. Нормализация (оптимизируемый тип)
        if normalization_type == 'Z-score':
            mean_per_bin = np.mean(linear_amp_spec, axis=1, keepdims=True)
            std_per_bin = np.std(linear_amp_spec, axis=1, keepdims=True)
            processed_spectrogram = (linear_amp_spec - mean_per_bin) / (std_per_bin + ZSCORE_EPS)
        elif normalization_type == 'PCEN':
            pcen_gain = params.get('pcen_gain', 0.98)
            pcen_bias = params.get('pcen_bias', 2.0)
            pcen_power = params.get('pcen_power', 0.5)
            pcen_time_constant = params.get('pcen_time_constant', 0.4)
            processed_spectrogram = librosa.pcen(
                linear_amp_spec * (2**15), sr=sr, hop_length=hop_length,
                gain=pcen_gain, bias=pcen_bias, power=pcen_power,
                time_constant=pcen_time_constant, eps=PCEN_DEFAULT_EPS
            )
        # --- НОВОЕ: Log нормализация ---
        elif normalization_type == 'Log':
            processed_spectrogram = np.log(linear_amp_spec + LOG_EPSILON)
        # --- КОНЕЦ НОВОГО ---
        elif normalization_type == 'None':
             processed_spectrogram = linear_amp_spec
        else: return None, None # Неизвестный тип

        if processed_spectrogram is None: return None, None
        if not np.all(np.isfinite(processed_spectrogram)): return None, None # Проверка на NaN/inf

        # 5. Подгонка размеров маски и спектрограммы
        n_frames_spec = processed_spectrogram.shape[1]
        if len(binary_mask) != n_frames_spec:
            if len(binary_mask) > n_frames_spec: binary_mask = binary_mask[:n_frames_spec]
            elif len(binary_mask) < n_frames_spec:
                if binary_mask.size > 0: binary_mask = np.pad(binary_mask, (0, n_frames_spec - len(binary_mask)), mode='edge')
                else: binary_mask = np.zeros(n_frames_spec, dtype=int)

        return processed_spectrogram, binary_mask
    except Exception as e:
        # traceback.print_exc() # Можно раскомментировать для детальной отладки
        return None, None

# --- Функции Метрик (без изменений) ---
def calculate_spectrogram_snr(spectrogram: np.ndarray, mask: np.ndarray, epsilon: float = 1e-6) -> float:
    if spectrogram is None or mask is None or spectrogram.size == 0 or mask.size == 0: return 0.0
    if spectrogram.shape[1] != mask.shape[0]: return 0.0
    signal_part = spectrogram[:, mask == 1]; noise_part = spectrogram[:, mask == 0]
    if signal_part.size == 0 or noise_part.size == 0: return 0.0
    s_mean = np.mean(signal_part); n_mean = np.mean(noise_part)
    # --- ИЗМЕНЕНИЕ: Log нормализация дает отрицательные значения, обрабатываем как Z-score ---
    if np.min(spectrogram) < 0 or normalization_type == 'Log': # Добавили 'Log'
        min_val = np.min(spectrogram); s_mean_shifted = s_mean - min_val; n_mean_shifted = n_mean - min_val
        snr = s_mean_shifted / n_mean_shifted if n_mean_shifted > epsilon else s_mean_shifted / epsilon
    # --- КОНЕЦ ИЗМЕНЕНИЯ ---
    else: # PCEN / None
        snr = s_mean / n_mean if n_mean > epsilon else s_mean / epsilon
    return snr if np.isfinite(snr) else 0.0

def calculate_active_freq_bins(spectrogram: np.ndarray, mask: np.ndarray, threshold_factor: float = ACTIVE_BIN_THRESHOLD_FACTOR) -> float:
    if spectrogram is None or mask is None or spectrogram.size == 0 or mask.size == 0: return 0.0
    if spectrogram.shape[1] != mask.shape[0]: return 0.0
    signal_frames = spectrogram[:, mask == 1]
    if signal_frames.size == 0: return 0.0
    s_mean_overall = np.mean(signal_frames); threshold = s_mean_overall * threshold_factor
    active_bins_per_frame = np.sum(signal_frames > threshold, axis=0)
    if active_bins_per_frame.size > 0:
        avg_active_bins = np.mean(active_bins_per_frame)
        return avg_active_bins if np.isfinite(avg_active_bins) else 0.0
    else: return 0.0

# --- Загрузка Списка Файлов (без изменений) ---
try:
    train_df = pd.read_csv(TRAIN_CSV_PATH)
    train_df['full_path'] = train_df['id'].apply(lambda x: audio_folder_path / f"{x}")
    valid_files_df = train_df[train_df['full_path'].apply(lambda p: p.exists())]
    if len(valid_files_df) >= AUDIO_SUBSET_SIZE:
        audio_files_subset = valid_files_df['full_path'].sample(AUDIO_SUBSET_SIZE, random_state=42).tolist()
        print(f"Используется {len(audio_files_subset)} случайных файлов для оценки.")
    elif len(valid_files_df) > 0:
         audio_files_subset = valid_files_df['full_path'].tolist()
         print(f"Warning: Найдено только {len(audio_files_subset)} файлов, используется вся выборка.")
    else:
        print("❌ ОШИБКА: Не найдено ни одного валидного аудиофайла для оценки!")
        audio_files_subset = []
except Exception as e:
    print(f"❌ ОШИБКА при загрузке списка файлов: {e}")
    audio_files_subset = []

# --- Целевая Функция для Optuna (v11 - Добавлена Log нормализация) ---

def objective_v11(trial):
    """ Целевая функция для оптимизации параметров обработки аудио (v11). """
    global normalization_type # <-- Нужно для функции SNR
    if not audio_files_subset:
        raise optuna.exceptions.TrialPruned("Нет аудиофайлов для оценки.")

    # 1. Предложение ОПТИМИЗИРУЕМЫХ параметров
    params = {}
    params['n_fft'] = FIXED_N_FFT

    hop_length_options = [h for h in [64, 96, 128] if h <= FIXED_N_FFT]
    if not hop_length_options: hop_length_options = [FIXED_N_FFT // 4]
    params['hop_length'] = trial.suggest_categorical('hop_length', hop_length_options)

    # --- ИЗМЕНЕНО: Добавлен 'Log' ---
    params['normalization_type'] = trial.suggest_categorical('normalization_type', ['None', 'Z-score', 'PCEN', 'Log'])
    normalization_type = params['normalization_type'] # Обновляем глобальную переменную для SNR
    # --- КОНЕЦ ИЗМЕНЕНИЯ ---

    params['pcen_gain'] = trial.suggest_float('pcen_gain', 0.1, 2.0, step=0.1)
    params['pcen_bias'] = trial.suggest_float('pcen_bias', 0.5, 10.0, step=0.5)
    params['pcen_power'] = trial.suggest_float('pcen_power', 0.1, 1.0, step=0.05)
    params['pcen_time_constant'] = trial.suggest_float('pcen_time_constant', 0.02, 0.5, step=0.02)

    # --- ДОБАВЛЯЕМ ФИКСИРОВАННЫЕ ПАРАМЕТРЫ МАСКИ ---
    params['threshold_smoothing_duration_s'] = FIXED_THRESHOLD_SMOOTHING_DURATION_S
    params['threshold_offset'] = FIXED_THRESHOLD_OFFSET
    params['min_absolute_rms_threshold'] = FIXED_MIN_ABSOLUTE_RMS_THRESHOLD
    params['apply_median_filter'] = FIXED_APPLY_MEDIAN_FILTER
    params['median_filter_ms'] = FIXED_MEDIAN_FILTER_MS
    # -------------------------------------------------

    # 2. Оценка на подвыборке файлов
    total_snr = 0.0; total_active_bins = 0.0; evaluated_files = 0; failed_files = 0
    total_files = len(audio_files_subset)

    for i, audio_path in enumerate(audio_files_subset):
        try:
            y, sr = sf.read(audio_path, dtype='float32')
            if sr != SAMPLE_RATE: y = librosa.resample(y, orig_sr=sr, target_sr=SAMPLE_RATE); sr = SAMPLE_RATE

            # --- ИСПОЛЬЗУЕМ НОВУЮ ФУНКЦИЮ РАСЧЕТА ПРИЗНАКОВ v11 ---
            processed_spec, binary_mask = calculate_features_v11_optuna(y, sr, params)
            # -------------------------------------------------

            if processed_spec is None or binary_mask is None: failed_files += 1; continue
            snr = calculate_spectrogram_snr(processed_spec, binary_mask)
            active_bins = calculate_active_freq_bins(processed_spec, binary_mask)
            if not math.isfinite(snr) or not math.isfinite(active_bins): failed_files += 1; continue
            total_snr += snr; total_active_bins += active_bins; evaluated_files += 1
        except Exception as e: failed_files += 1; continue

    # 3. Расчет и возврат итоговой метрики с учетом штрафа
    if evaluated_files == 0: return -1.0
    avg_snr = total_snr / evaluated_files; avg_active_bins = total_active_bins / evaluated_files
    base_score = max(0, avg_snr) * (avg_active_bins + 0.1)
    fail_ratio = failed_files / total_files; penalty = fail_ratio * FAILED_FILE_PENALTY_FACTOR
    final_score = base_score * (1.0 - penalty)

    # Сохраняем доп. информацию
    try:
        trial.set_user_attr("avg_snr", float(avg_snr)); trial.set_user_attr("avg_active_bins", float(avg_active_bins))
        trial.set_user_attr("base_score", float(base_score)); trial.set_user_attr("fail_ratio", float(fail_ratio))
        trial.set_user_attr("penalty", float(penalty)); trial.set_user_attr("evaluated_files", int(evaluated_files))
        trial.set_user_attr("failed_files", int(failed_files))
    except Exception as e_attr: pass

    return final_score if math.isfinite(final_score) else 0.0

# --- Запуск Оптимизации ---
if __name__ == "__main__":
    if not audio_files_subset:
        print("Невозможно запустить оптимизацию: нет доступных аудиофайлов.")
    else:
        # --- ИЗМЕНЕНО ИМЯ ---
        study_name = f"morse_param_optimization_v13_fixed_mask_nfft_{FIXED_N_FFT}" # Новое имя v11
        # --- КОНЕЦ ИЗМЕНЕНИЯ ---
        storage_name = f"sqlite:///{study_name}.db"
        study = optuna.create_study(
            study_name=study_name,
            storage=storage_name,
            load_if_exists=True,
            direction='maximize'
        )

        print(f"\nЗапуск/продолжение исследования '{study_name}'...")
        print(f"Используется ФИКСИРОВАННЫЙ n_fft = {FIXED_N_FFT}")
        print("Используются ФИКСИРОВАННЫЕ параметры адаптивной маски:")
        print(f"  - threshold_smoothing_duration_s: {FIXED_THRESHOLD_SMOOTHING_DURATION_S:.2f}")
        print(f"  - threshold_offset: {FIXED_THRESHOLD_OFFSET:.4f}")
        print(f"  - min_absolute_rms_threshold: {FIXED_MIN_ABSOLUTE_RMS_THRESHOLD:.4f}")
        print(f"  - apply_median_filter: {FIXED_APPLY_MEDIAN_FILTER}")
        print(f"  - median_filter_ms: {FIXED_MEDIAN_FILTER_MS}")
        # --- ИЗМЕНЕНО ОПИСАНИЕ ---
        print(f"\nOptuna будет подбирать: hop_length, normalization_type ('None', 'Z-score', 'PCEN', 'Log') и параметры PCEN.")
        # --- КОНЕЦ ИЗМЕНЕНИЯ ---
        print(f"Применяется штраф за ошибки обработки файлов (фактор={FAILED_FILE_PENALTY_FACTOR}).")
        print(f"Будет выполнено {N_TRIALS} попыток (или дозавершено до {N_TRIALS}).")
        print(f"Результаты сохраняются в: {storage_name}")

        with warnings.catch_warnings():
             warnings.simplefilter("ignore")
             # --- ИЗМЕНЕН ВЫЗОВ ---
             study.optimize(objective_v11, n_trials=N_TRIALS, show_progress_bar=True)
             # --- КОНЕЦ ИЗМЕНЕНИЯ ---

        # --- Вывод Результатов ---
        print("\n--- Оптимизация Завершена ---")
        print(f"Количество завершенных попыток: {len(study.trials)}")
        completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE and t.value is not None and math.isfinite(t.value)]
        if not completed_trials:
             print("\nНе найдено ни одного успешно завершенного испытания с валидным score.")
        else:
            best_trial = max(completed_trials, key=lambda t: t.value)
            if best_trial.value <= 0:
                 print(f"\nНе удалось найти триал с положительным значением метрики. Лучшее значение: {best_trial.value:.4f}")
                 print("Параметры лучшего (не положительного) триала:")
            else:
                print(f"\nЛучшее значение метрики (final_score): {best_trial.value:.4f}")
                print(f"Лучшие ОПТИМИЗИРОВАННЫЕ параметры (для фикс. n_fft={FIXED_N_FFT} и фикс. маски):")

            for key, value in best_trial.params.items(): print(f"  {key}: {value}")
            print("\nФиксированные параметры маски:")
            print(f"  threshold_smoothing_duration_s: {FIXED_THRESHOLD_SMOOTHING_DURATION_S:.2f}")
            print(f"  threshold_offset: {FIXED_THRESHOLD_OFFSET:.4f}")
            print(f"  min_absolute_rms_threshold: {FIXED_MIN_ABSOLUTE_RMS_THRESHOLD:.4f}")
            print(f"  apply_median_filter: {FIXED_APPLY_MEDIAN_FILTER}")
            print(f"  median_filter_ms: {FIXED_MEDIAN_FILTER_MS}")

            print("\nМетрики лучшего триала:")
            if best_trial.user_attrs:
                for key, value in best_trial.user_attrs.items():
                    if isinstance(value, float): print(f"  {key}: {value:.4f}")
                    else: print(f"  {key}: {value}")
            else: print("  User attributes не найдены.")

            print("\nПопытка построения графиков визуализации...")
            try:
                fig_history = optuna.visualization.plot_optimization_history(study)
                fig_history.show()
                fig_importance = optuna.visualization.plot_param_importances(study)
                fig_importance.show()
                print("Графики успешно созданы.")
            except Exception as e: print(f"\nОшибка при построении графиков: {e}")
            except ImportError: print("\nДля построения графиков Optuna установите plotly: pip install plotly")

[I 2025-04-22 13:34:25,552] Using an existing study with name 'morse_param_optimization_v13_fixed_mask_nfft_512' instead of creating a new one.


Используется 50 случайных файлов для оценки.

Запуск/продолжение исследования 'morse_param_optimization_v13_fixed_mask_nfft_512'...
Используется ФИКСИРОВАННЫЙ n_fft = 512
Используются ФИКСИРОВАННЫЕ параметры адаптивной маски:
  - threshold_smoothing_duration_s: 1.40
  - threshold_offset: 0.0400
  - min_absolute_rms_threshold: 0.0500
  - apply_median_filter: False
  - median_filter_ms: 0

Optuna будет подбирать: hop_length, normalization_type ('None', 'Z-score', 'PCEN', 'Log') и параметры PCEN.
Применяется штраф за ошибки обработки файлов (фактор=0.2).
Будет выполнено 200 попыток (или дозавершено до 200).
Результаты сохраняются в: sqlite:///morse_param_optimization_v13_fixed_mask_nfft_512.db


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2025-04-22 13:34:27,240] Trial 706 finished with value: 366.4678515150322 and parameters: {'hop_length': 128, 'normalization_type': 'PCEN', 'pcen_gain': 0.7000000000000001, 'pcen_bias': 9.0, 'pcen_power': 1.0, 'pcen_time_constant': 0.22}. Best is trial 252 with value: 366.5926157942296.
[I 2025-04-22 13:34:28,933] Trial 707 finished with value: 360.93408940874576 and parameters: {'hop_length': 64, 'normalization_type': 'PCEN', 'pcen_gain': 0.8, 'pcen_bias': 9.0, 'pcen_power': 0.4, 'pcen_time_constant': 0.22}. Best is trial 252 with value: 366.5926157942296.
[I 2025-04-22 13:34:29,949] Trial 708 finished with value: 361.960326236816 and parameters: {'hop_length': 128, 'normalization_type': 'PCEN', 'pcen_gain': 0.8, 'pcen_bias': 9.0, 'pcen_power': 0.9500000000000001, 'pcen_time_constant': 0.13999999999999999}. Best is trial 252 with value: 366.5926157942296.
[I 2025-04-22 13:34:30,917] Trial 709 finished with value: 366.35982744083276 and parameters: {'hop_length': 128, 'normalization

KeyboardInterrupt: 

In [ ]:
# Ячейка для визуализации разделения Сигнал/Шум (v5 - Исправлена NameError)

import matplotlib.pyplot as plt
import librosa
import librosa.display
import numpy as np
import soundfile as sf
# from scipy.signal import medfilt # Не используем медиану
from pathlib import Path
import warnings
import ipywidgets as widgets
from ipywidgets import interact, Layout, VBox
from IPython.display import display, clear_output

# --- Параметры ---
params = {
    'n_fft': 1024,      # Из лучших параметров Optuna
    'hop_length': 192,  # Из лучших параметров Optuna
}
SAMPLE_RATE = 8000
# --- Параметры Адаптивного Порога (на основе Среднего) ---
MOVING_AVG_DURATION_S = 0.6 # Длительность окна для среднего (в секундах)
THRESHOLD_OFFSET = 0.01    # Насколько RMS должен быть выше локального среднего

# --- Выбор файла ---
# Убедитесь, что 'audio_folder_path' и 'valid_files_df' определены
# DATA_DIR = Path("./")
# AUDIO_FOLDER_NAME = "morse_dataset/morse_dataset"
# audio_folder_path = DATA_DIR / AUDIO_FOLDER_NAME

if 'valid_files_df' in locals() and not valid_files_df.empty:
     example_file_path = valid_files_df['full_path'].iloc[0] # Пример
     # example_file_id = "YOUR_FILE_ID.opus" # Укажите конкретный ID
     # example_file_path = audio_folder_path / example_file_id
     print(f"Используется файл: {example_file_path}")
else:
     example_file_path = None
     print("❌ ОШИБКА: Не найдены аудиофайлы. Укажите путь к файлу вручную.")

# --- Глобальные переменные для интерактивности ---
current_rms_data = {'rms': None, 'times': None, 'sr': SAMPLE_RATE, 'hop': params['hop_length'], 'n_fft': params['n_fft']}
plot_output_adaptive = Output() # Область для вывода графика

# --- Функция расчета адаптивной маски (v4 - Скользящее Среднее) ---
def calculate_adaptive_mask_mean(rms_values, sr, hop_length, moving_avg_duration_s, threshold_offset):
    """Рассчитывает адаптивный порог (на основе скользящего среднего) и бинарную маску."""
    if rms_values is None or rms_values.size == 0:
        return None, None

    moving_avg_len_frames = int(moving_avg_duration_s * sr / hop_length)
    if moving_avg_len_frames % 2 == 0: moving_avg_len_frames += 1

    if moving_avg_len_frames >= len(rms_values):
         print(f"Warning: Окно ({moving_avg_len_frames}) больше или равно длине данных ({len(rms_values)}). Используется среднее по всему сигналу.")
         local_mean = np.full_like(rms_values, np.mean(rms_values))
    else:
        try:
            window = np.ones(moving_avg_len_frames) / moving_avg_len_frames
            local_mean = np.convolve(rms_values, window, mode='same')
            if len(local_mean) != len(rms_values):
                 print(f"Warning: Длина результата convolve ({len(local_mean)}) не совпадает с исходной ({len(rms_values)}).")
                 if len(local_mean) > len(rms_values): local_mean = local_mean[:len(rms_values)]
                 else: pad_width = len(rms_values) - len(local_mean); local_mean = np.pad(local_mean, (pad_width // 2, pad_width - pad_width // 2), mode='mean')
        except Exception as e_conv:
            print(f"Ошибка при расчете скользящего среднего: {e_conv}. Используется среднее по всему сигналу.")
            local_mean = np.full_like(rms_values, np.mean(rms_values))

    adaptive_threshold = local_mean + threshold_offset
    binary_mask = (rms_values >= adaptive_threshold).astype(int)
    return adaptive_threshold, binary_mask

# --- Функция визуализации (v5 - Исправлена NameError, добавлен offset) ---
def plot_signal_noise_separation_v5( # Переименована в v5
    rms_values, times, adaptive_threshold, binary_mask, offset, title_suffix="" # Добавлен параметр offset
    ):
    """Визуализирует RMS, адаптивный порог (среднее) и маску."""
    if rms_values is None or times is None or adaptive_threshold is None or binary_mask is None:
        print("Ошибка: Недостаточно данных для построения графика.")
        return

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True) # 2 строки

    # График 1: RMS и Адаптивный Порог (Среднее)
    if len(times) > 0:
        axes[0].plot(times, rms_values, label='RMS Энергия', color='purple', alpha=0.6, linewidth=1.5)
        # --- ИСПРАВЛЕНИЕ МЕТКИ ---
        axes[0].plot(times, adaptive_threshold, color='red', linestyle='--',
                     label=f'Адаптивный порог (Mean + {offset:.4f})') # Используем переданный offset
        # --- КОНЕЦ ИСПРАВЛЕНИЯ ---
        axes[0].set_title(f'RMS Энергия и Адаптивный Порог (на основе Среднего) {title_suffix}')
        axes[0].set_ylabel('RMS')
        axes[0].legend(fontsize='small')
        axes[0].grid(True, linestyle='--', alpha=0.6)
        axes[0].set_xlim(times[0], times[-1])
        axes[0].fill_between(times, 0, adaptive_threshold, color='red', alpha=0.1, label='Область ниже порога')
    else:
        axes[0].set_title('RMS Энергия и Адаптивный Порог (Ошибка: нет данных)')

    # График 2: Маска Сигнал/Шум
    mask_image = binary_mask[np.newaxis, :]
    if len(times) > 0:
        img = axes[1].imshow(mask_image, aspect='auto', cmap='gray_r',
                             interpolation='nearest',
                             extent=[times[0], times[-1], 0, 1])
        axes[1].set_title('Маска Сигнал (1) / Шум (0) - Результат Адаптивного Порога (Среднее)')
        axes[1].set_xlabel('Время (с)')
        axes[1].set_yticks([])
    else:
         axes[1].set_title('Маска Сигнал (1) / Шум (0) (Ошибка: нет данных)')

    plt.tight_layout()
    plt.show()

# --- Функция-обертка для интерактивного вызова (v5) ---
def update_adaptive_plot_mean_v5(moving_avg_duration, offset): # Переименована в v5
    """Пересчитывает маску (по среднему) и обновляет график при изменении виджетов."""
    global current_rms_data, plot_output_adaptive
    rms_vals = current_rms_data.get('rms')
    times = current_rms_data.get('times')
    sr = current_rms_data.get('sr')
    hop = current_rms_data.get('hop')

    if rms_vals is None:
        with plot_output_adaptive:
            clear_output(wait=True)
            print("Сначала загрузите и обработайте RMS данные.")
        return

    adaptive_threshold, binary_mask = calculate_adaptive_mask_mean(
        rms_vals, sr, hop, moving_avg_duration, offset
    )

    with plot_output_adaptive:
        clear_output(wait=True)
        if adaptive_threshold is not None:
             print(f"Параметры: Окно среднего={moving_avg_duration:.2f}с, Отступ={offset:.4f}")
             # --- ИЗМЕНЕН ВЫЗОВ ---
             plot_signal_noise_separation_v5( # Вызываем v5
                 rms_vals, times, adaptive_threshold, binary_mask,
                 offset=offset, # Передаем offset
                 title_suffix=f" (Moving Avg Window: {moving_avg_duration:.2f}s, Offset: {offset:.4f})"
             )
             # --- КОНЕЦ ИЗМЕНЕНИЯ ---
        else:
             print("Не удалось рассчитать адаптивный порог.")


# --- Загрузка и первичный расчет RMS ---
def load_and_calculate_initial_rms(audio_path, sr, n_fft, hop_length):
    """Загружает аудио и рассчитывает RMS (без изменений)."""
    global current_rms_data
    if not audio_path or not audio_path.exists():
        print(f"Файл не найден: {audio_path}")
        current_rms_data = {'rms': None, 'times': None, 'sr': sr, 'hop': hop_length, 'n_fft': n_fft}
        return False
    try:
        y, loaded_sr = sf.read(audio_path, dtype='float32')
        if loaded_sr != sr:
            y = librosa.resample(y, orig_sr=loaded_sr, target_sr=sr)
        rms_values = librosa.feature.rms(y=y, frame_length=n_fft, hop_length=hop_length)[0]
        times = librosa.times_like(rms_values, sr=sr, hop_length=hop_length, n_fft=n_fft)
        if len(times) > len(rms_values): times = times[:len(rms_values)]
        elif len(times) < len(rms_values): rms_values = rms_values[:len(times)]
        current_rms_data['rms'] = rms_values
        current_rms_data['times'] = times
        print(f"RMS данные для {audio_path.name} загружены и рассчитаны.")
        return True
    except Exception as e:
        print(f"Ошибка загрузки или расчета RMS для {audio_path.name}: {e}")
        current_rms_data = {'rms': None, 'times': None, 'sr': sr, 'hop': hop_length, 'n_fft': n_fft}
        return False

# --- Создание интерактивных виджетов (без изменений) ---
style = {'description_width': '150px'}
w_moving_avg_duration = widgets.FloatSlider(
    min=0.1, max=2.0, step=0.05, value=MOVING_AVG_DURATION_S,
    description='Окно среднего (с):', style=style, continuous_update=False, readout_format='.2f'
)
w_threshold_offset = widgets.FloatSlider(
    min=0.0, max=0.1, step=0.001, value=THRESHOLD_OFFSET,
    description='Отступ от среднего:', style=style, continuous_update=False, readout_format='.4f'
)

# --- Отображение и запуск (v5) ---
print("--- Интерактивный подбор Адаптивного Порога (на основе Среднего) v5 ---")
if example_file_path:
    load_and_calculate_initial_rms(example_file_path, SAMPLE_RATE, params['n_fft'], params['hop_length'])

    interactive_plot = interactive(update_adaptive_plot_mean_v5, # Вызываем v5
                                   moving_avg_duration=w_moving_avg_duration,
                                   offset=w_threshold_offset)
    controls = VBox(interactive_plot.children[:-1], layout=Layout(flex_flow='row wrap'))
    # plot_output_adaptive = interactive_plot.children[-1] # Область вывода графика

    display(controls, plot_output_adaptive)

    # Вызываем первичную отрисовку с начальными значениями
    update_adaptive_plot_mean_v5(w_moving_avg_duration.value, w_threshold_offset.value)
else:
    print("Не выбран файл для анализа.")

Используется файл: morse_dataset\morse_dataset\1.opus
--- Интерактивный подбор Адаптивного Порога (на основе Среднего) v5 ---
RMS данные для 1.opus загружены и рассчитаны.


Output()